# Cross-Wearer Gesture Recognition — 1D CNN with LOUO Evaluation

This notebook trains a **1D convolutional neural network** (a CNN that slides filters along the time axis of a multi-channel signal) to recognise hand gestures from wearable IMU and flex-sensor data. It is built around a single question: *how well does the trained model work on a brand-new wearer it has never seen during training?*

## Why this is harder than it sounds

A model trained and tested on the same set of people will look brilliant — usually 95 %+ accuracy — because the test data and training data come from the same wearers. Once you swap in someone new, accuracy can drop to 60 % or worse. The reason is that the network has quietly learned per-wearer quirks: how the glove sits on a hand, how forcefully a person moves, where their hand rests, what each sensor's baseline reads. Those quirks shift on a new user, and the learned features stop working.

## How this notebook measures honestly

It uses **Leave-One-User-Out (LOUO) cross-validation**. In each fold, every trial from one user is held out as the test set, and the model is trained on everyone else. The reported number is the mean accuracy across all folds — this is what you should actually expect on a wearer who is not in your dataset.

## Three changes that move the needle

1. **Per-trial normalisation** — each trial is rescaled (z-score or min-max, configurable) so per-wearer offset and scale are erased before the model ever sees the data.
2. **Wearer-style augmentation** — during training, copies of each trial are randomly rotated (different glove angle), gain-scaled (different sensor sensitivity), and DC-shifted (different baselines). The CNN sees many synthetic "wearers" per real one.
3. **Per-user inner-validation** — when EarlyStopping needs a validation set, the notebook holds out a *different* user instead of a random slice of training trials. This prevents the model from indirectly tuning itself on the wearer it will be scored on.

## How to use this notebook

1. **Open Section 2** and edit `LOUO_PATHS` to list the folders you want to include. Each entry should point to one user's gesture-set folder (e.g. `…/NewTestData/1_Alan/Dynamic`). User names come from the parent directory.
2. **Run every cell top to bottom.** Section 6 prints the LOUO accuracy; Section 7 retrains one model on (almost) all of the data; Sections 8 and 9 optionally save the model and a PDF summary.

Every knob — preprocessing, augmentation, training, output paths — lives in Section 2 so you only ever edit one cell.


## 1. Imports

Standard scientific-Python tooling (NumPy, pandas, SciPy, scikit-learn) plus Keras for the neural-network model. Every later section assumes these are loaded.


In [14]:
# Core scientific-Python stack used throughout the notebook.
import os
import glob
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Signal processing (low-pass filter) and ML helpers (scaler + label encoder).
from scipy import signal as scipy_signal
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Used only by the PDF report cell (Section 9) to render charts.
import matplotlib
matplotlib.use('Agg')   # render off-screen, no display required
import matplotlib.pyplot as plt

# Keras layers and the Sequential model wrapper. Variant builders in
# Section 2a import a few extra layers (BatchNorm, GAP, LSTM, etc.).
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

np.random.seed(42)
pd.set_option('display.max_columns', 20)
print('Imports OK.')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from scipy.spatial.distance import pdist, squareform


Imports OK.


## 2. Configuration

Every parameter that controls the notebook lives in the cell below — data paths, sensor selection, preprocessing, augmentation, training, evaluation strategy, and output toggles. The remaining sections only *read* these variables, so this is the only cell you edit when tuning a run.

**Most common edits**

- **`LOUO_PATHS`** *(sub-section 1)* — the list of user folders to include. Each entry is a full path to one user's gesture-set folder. User identity is taken from the parent directory name. Comment out a line to exclude that user from this run.
- **`PER_TRIAL_NORM`** *(sub-section 6)* — `'zscore'`, `'minmax'`, or `'none'`. Controls the per-wearer offset / scale removal stage.
- **`AUGMENT_CONFIG`** *(sub-section 4)* — which augmentations are enabled and how strong they are. The last three keys (`rotate_accel`, `channel_gain`, `channel_dc`) are the wearer-style ones.
- **`LOUO_INNER_VAL`** *(sub-section 7)* — how the EarlyStopping validation set is built: `'user'` (recommended), `'random'`, or `'none'`.
- **`SAVE_DEPLOYMENT_MODEL`** / **`GENERATE_PDF_REPORT`** *(sub-section 8)* — turn off if you only want the LOUO numbers.

The only configurable variable that lives outside this cell is `VARIANTS` (the list of network architectures), which sits in Section 2a immediately below because it depends on Keras layer imports.
- **`EXCLUDE_CLASSES`** — gesture-folder names to drop at load time (e.g. noisy or never-used labels). Leave empty to keep all.


In [15]:
# ════════════════════════════════════════════════════════════════════════════
#                 SECTION 2 — MASTER CONFIGURATION
# ════════════════════════════════════════════════════════════════════════════
#  Every knob that controls the pipeline lives in this single cell. The rest
#  of the notebook only *reads* these variables.
#
#  Sub-sections:
#     (1) LOUO data — list every <user>/Dynamic folder you want to include
#     (2) Sensor / segment selection
#     (3) Preprocessing
#     (4) Augmentation (incl. wearer-style augmenters)
#     (5) Training (epochs / batch size / early stopping)
#     (6) Per-trial normalisation
#     (7) LOUO inner-validation strategy
#     (8) Final-retrain hold-out user (used for the saved deployment model)
#     (9) Model output paths
#     (10) Completion notifications (desktop / beep / push)
#
#  NOTE: VARIANTS (architectures to compare) is defined in Section 2a
#        because it depends on builder functions defined above it.
# ════════════════════════════════════════════════════════════════════════════
from pathlib import Path

# ─────────────────────── (1) LOUO DATA ───────────────────────────────────
# List every folder you want to include in LOUO evaluation. Each entry must
# point to a single user's gesture-set directory (typically <user>/Dynamic).
# The user identity is taken from the PARENT directory name. Comment out a
# line to exclude that user from this run.
LOUO_PATHS = [

    # # # '/home/jestin/ThesisRepo/ML/NewTrainingData/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/1_Alan/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/2_Alex/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/3_Anghad/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/4_Daniel/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/5_Georgia/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/6_Harry/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/7_Henry/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/8_Jestin/Static',
    # # '/home/jestin/ThesisRepo/ML/NewTestData/9_Joselyn/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/10_Josh/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/11_Mansh/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/12_Marcus/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/13_Nat/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/14_StephenV2/Static',
    # # '/home/jestin/ThesisRepo/ML/NewTestData/15_Tash/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/16_Bella/Static',
    # '/home/jestin/ThesisRepo/ML/NewTestData/17_Raquel/Static',


    # '/home/jestin/ThesisRepo/ML/NewTrainingData/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/1_Alan/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/2_Alex/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/3_Anghad/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/4_Daniel/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/5_Georgia/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/6_Harry/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/7_Henry/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/8_Jestin/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/9_Joselyn/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/10_Josh/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/11_Mansh/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/12_Marcus/Dynamic',
    # '/home/jestin/ThesisRepo/ML/NewTestData/13_Nat/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/14_StephenV2/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/15_Tash/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/16_Bella/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/17_Raquel/Dynamic',
    '/home/jestin/ThesisRepo/ML/NewTestData/18_Bridgette/Dynamic'
]

# Classes (label-folder names) to drop entirely. Trials with these labels
# are skipped at load time so they never appear in training, evaluation,
# diagnostics, or the label encoder. Leave the list empty to keep all
# classes. Names must match the folder names exactly (case-sensitive).
EXCLUDE_CLASSES = [
    # 'Drum_Roll',
    'Double_Nothing',
]
# EXCLUDE_CLASSES = None
# ─────────────────────── (2) SENSOR / SEGMENT SELECTION ──────────────────
USE_LEFT_HAND  = True
USE_RIGHT_HAND = True
USE_YPR        = True   # yaw / pitch / roll (or heading/pitch/roll for wrist)
USE_QUAT       = False  # quaternion w/x/y/z
USE_ACCEL      = True   # ax / ay / az
USE_FLEX       = True   # mcp_flex / pip_flex (fingers only)

USE_WRIST  = True
USE_PALM   = True
USE_THUMB  = True
USE_INDEX  = True
USE_MIDDLE = True
USE_RING   = True
USE_PINKY  = True

# ─────────────────────── (3) PREPROCESSING ───────────────────────────────
APPLY_BUTTERWORTH      = True
BUTTERWORTH_CUTOFF_HZ  = 10.0
BUTTERWORTH_ORDER      = 4
SAMPLING_RATE_HZ       = 30.0
GESTURE_DURATION_S     = 3.0
USE_SECONDS            = 3.0      # seconds from start of each trial to keep
# Clamp + derive timestep counts. RESAMPLE_TO_N_STEPS auto-tracks the trim.
USE_SECONDS            = float(min(max(USE_SECONDS, 0.0), GESTURE_DURATION_S))
_USE_N_SAMPLES         = max(2, int(round(USE_SECONDS * SAMPLING_RATE_HZ)))
RESAMPLE_TO_N_STEPS    = _USE_N_SAMPLES
print(f'[config] Using first {USE_SECONDS:.2f} s of each trial '
      f'(≈ {_USE_N_SAMPLES} samples → RESAMPLE_TO_N_STEPS={RESAMPLE_TO_N_STEPS})')
# Downstream scaler. Skipped automatically when PER_TRIAL_NORM != 'none'
# (per-trial normalisation already puts every trial on a common scale,
# so a second rescaling would just shuffle the numbers without adding info).
NORMALISATION          = 'minmax'   # 'standard' | 'minmax' | None
RANDOM_STATE           = 42         # global seed for augmentation + model init

# ─────────────────────── (4) AUGMENTATION ────────────────────────────────
AUGMENT_TRAINING_DATA          = True
AUGMENTATION_COPIES_PER_SAMPLE = 2
AUGMENTATION_RANDOM_SEED       = 42

# Per-augmenter on/off + parameter dict. The last three keys
# (rotate_accel / channel_gain / channel_dc) are wearer-style augmenters
# specifically designed to improve cross-wearer generalisation.
AUGMENT_CONFIG = {
    'time_shift':      {'enabled': False,  'apply_prob': 0.7, 'max_shift_steps': 3, 'fill_mode': 'edge'},
    'time_warp':       {'enabled': False,  'apply_prob': 0.5, 'speed_range': (0.90, 1.10)},
    'time_mask':       {'enabled': False,  'apply_prob': 0.5, 'max_masks': 2, 'max_mask_size': 3, 'fill_mode': 'zero'},
    'gaussian_noise':  {'enabled': False,  'apply_prob': 1.0, 'std_ratio': 0.02},
    'amplitude_scale': {'enabled': True,  'apply_prob': 0.5, 'scale_range': (0.95, 1.05)},
    'baseline_offset': {'enabled': True,  'apply_prob': 0.5, 'offset_std_ratio': 0.02},
    'channel_dropout': {'enabled': False,  'apply_prob': 0.3, 'drop_fraction': 0.03, 'fill_mode': 'zero'},
    # Only enable left/right swap if the class label is symmetric under that swap
    'left_right_swap': {'enabled': False, 'apply_prob': 0.5},

    # ── Wearer-style augmenters (training-time only) ──────────────────
    'rotate_accel':    {'enabled': False,  'apply_prob': 0.7, 'max_deg': 10.0},
    'channel_gain':    {'enabled': False,  'apply_prob': 0.5, 'scale_range': (0.85, 1.15)},
    'channel_dc':      {'enabled': False,  'apply_prob': 0.5, 'offset_std_ratio': 0.10},
}

# ─────────────────────── (5) TRAINING ────────────────────────────────────
TRAIN_EPOCHS             = 40
TRAIN_BATCH_SIZE         = 16
TRAIN_VERBOSE            = 1
USE_EARLY_STOPPING       = True
EARLY_STOPPING_PATIENCE  = 8

# ─────────────────────── (6) PER-TRIAL NORMALISATION ────────────────────
# Each trial is rescaled along its time axis (per channel) using ONLY its
# own statistics before anything else. Removes per-wearer DC offset and
# amplitude scale — the single biggest lever for cross-user accuracy.
#   "zscore" → subtract per-channel mean, divide by per-channel std
#              (each channel of each trial ends up with mean 0 / std 1)
#   "minmax" → subtract per-channel min, divide by per-channel range,
#              then scale to PER_TRIAL_MINMAX_RANGE (each channel of each
#              trial spans that range)
#   "none"   → skip per-trial normalisation; the downstream scaler
#              (NORMALISATION above) takes over
#
#  Whenever PER_TRIAL_NORM != 'none', the downstream MinMax/Standard
#  scaler is automatically skipped so we never normalise twice. To run
#  a dataset-wide normalisation baseline (one set of per-channel stats
#  fit on the training pool only), set PER_TRIAL_NORM='none' and pick
#  NORMALISATION='standard' or 'minmax' in sub-section (3) instead.
PER_TRIAL_NORM = 'none'             # 'zscore' | 'minmax' | 'none'
PER_TRIAL_MINMAX_RANGE = (0.0, 1.0)   # only used when PER_TRIAL_NORM=='minmax'

# ─────────────────────── (7) LOUO + INNER-VALIDATION STRATEGY ────────────
# Which variants to evaluate. Must be names from VARIANTS (Section 2a).
LOUO_VARIANTS  = [
    'Baseline',
    'Shallow',
    'Deep',
    'BN_GAP',
    'WideKernel',
    'CNN_LSTM',
    'CNN_BiLSTM',
    # 'BN_GAP_Wide',
    # 'WideKernel_GAP',
    # 'BN_GAP_v2',
    # 'ResNet1D',
    # 'MultiScale',

    ]   # e.g. ['Deep', 'BN_GAP_Wide']



# How to build the inner-validation set used by EarlyStopping inside each fold.
#   'user'   = deterministically rotate one OTHER user out each fold
#              → monitor val_loss on a different held-out wearer.
#                RECOMMENDED.
#   'random' = legacy behaviour, random 15% slice of the training pool.
#   'none'   = no validation set; EarlyStopping monitors training loss.
LOUO_INNER_VAL       = 'user'
LOUO_INNER_VAL_SPLIT = 0.15           # only used when LOUO_INNER_VAL='random'
# ─────────────────────── (8) FINAL-RETRAIN HOLD-OUT ──────────────────────
# After LOUO measures generalisation, we retrain ONE deployment model on
# every user EXCEPT this one (so we can still report a clean held-out
# accuracy for the saved model). Set to None to train on every user.
RUN_FINAL_RETRAIN          = False  # set True to retrain a single deployment model on all users
FINAL_RETRAIN_HELD_OUT_USER = None   # e.g. '8_Jestin'  (None → train on all)
FINAL_RETRAIN_VARIANT       = 'BN_GAP_Wide' # which architecture to save

# Persisted-artefact toggles. The LOUO evaluation and final retrain
# always run; these only control what gets written to disk.
#   SAVE_DEPLOYMENT_MODEL → .keras model + scaler + labels + CSV log
#   GENERATE_PDF_REPORT   → PDF summary of config + LOUO + retrain
SAVE_DEPLOYMENT_MODEL = False
GENERATE_PDF_REPORT   = True

# ─────────────────────── (9) MODEL OUTPUT PATHS ──────────────────────────
MODEL_OUTPUT_DIR    = Path('/home/jestin/ThesisRepo/ML/NewReports/DynamicReports')
EXPERIMENT_LOG_PATH = MODEL_OUTPUT_DIR / 'experiment_results.csv'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────── (10) COMPLETION NOTIFICATIONS ───────────────────
# LOUO can take a long time. Section 10 (the final cell) fires whichever
# of these channels you have enabled, so you can walk away during the run.
#   NOTIFY_DESKTOP → Ubuntu toast via `notify-send` (libnotify). Requires
#                    the `libnotify-bin` package (already on most desktops).
#   NOTIFY_BEEP    → short system bell from inside the notebook. Plays
#                    even with headphones on, useful when you're nearby.
#   NOTIFY_PUSH    → off-device push via https://ntfy.sh. Free, no signup —
#                    install the ntfy app on your phone, subscribe to
#                    NOTIFY_NTFY_TOPIC, and you'll get a push when the run
#                    finishes. The topic acts like a password; pick a long,
#                    hard-to-guess string.
NOTIFY_DESKTOP   = True
NOTIFY_BEEP      = True
NOTIFY_PUSH      = False    # set to True after you've picked a topic below
NOTIFY_NTFY_TOPIC = 'jestin-thesis-louo-CHANGE-ME-to-something-unique'

print(f'Configured {len(LOUO_PATHS)} LOUO user folder(s).')


[config] Using first 3.00 s of each trial (≈ 90 samples → RESAMPLE_TO_N_STEPS=90)
Configured 13 LOUO user folder(s).


## 2a. Architecture variants

`VARIANTS` is a list of `(name, description, builder)` tuples — one per candidate CNN architecture. Each `builder(sequence_length, n_channels, n_classes)` returns a compiled Keras model.

Two settings in Section 2 reference these names by string:

- **`LOUO_VARIANTS`** — which architectures the LOUO loop evaluates.
- **`FINAL_RETRAIN_VARIANT`** — which architecture is retrained on the full pool and saved as the deployment model.

The names must match the first element of a tuple in `VARIANTS` below. Add a new architecture by appending another tuple; remove one by deleting the line.


In [16]:
from tensorflow.keras.layers import (
    BatchNormalization, GlobalAveragePooling1D, Activation, Input,
    LSTM, Bidirectional,
)
from tensorflow.keras.models import Sequential

# from tensorflow.keras.optimizers import Adam




def build_baseline(seq_len, n_chan, n_classes):
    """Original architecture from the notebook."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='Baseline')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_shallow(seq_len, n_chan, n_classes):
    """Single Conv block — fewer params, faster to train."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 5, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='Shallow')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_deep(seq_len, n_chan, n_classes):
    """Three Conv blocks — more representational depth."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, activation='relu'),
        MaxPooling1D(2),
        Conv1D(128, 3, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.4),
        Dense(n_classes, activation='softmax'),
    ], name='Deep')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_bn_gap(seq_len, n_chan, n_classes):
    """BatchNorm + GlobalAveragePooling — typically better generalisation, fewer params."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(128, 3, padding='same'),
        BatchNormalization(), Activation('relu'),
        GlobalAveragePooling1D(),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='BN_GAP')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_wide_kernel(seq_len, n_chan, n_classes):
    """Wider kernels — larger temporal receptive field per layer."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 8, activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 8, activation='relu'),
        MaxPooling1D(2),
        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='WideKernel')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_cnn_lstm(seq_len, n_chan, n_classes):
    """CNN feature extractor → LSTM temporal model.

    Two Conv1D blocks pull out local temporal features, then a single LSTM
    consumes the resulting (T', F) sequence and a final Dense head classifies.
    Conv layers use padding='same' so MaxPooling controls the temporal
    downsampling explicitly.
    """
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, padding='same', activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, padding='same', activation='relu'),
        MaxPooling1D(2),
        Dropout(0.3),
        LSTM(64, return_sequences=False),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='CNN_LSTM')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_cnn_bilstm(seq_len, n_chan, n_classes):
    """CNN feature extractor → Bidirectional LSTM temporal model.

    Same Conv front-end as CNN_LSTM but the recurrent stage is bidirectional,
    so each timestep's representation sees both past and future context within
    the trial — useful for gestures where the discriminative motion can sit
    anywhere along the trial.
    """
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 4, padding='same', activation='relu'),
        MaxPooling1D(2),
        Conv1D(64, 4, padding='same', activation='relu'),
        MaxPooling1D(2),
        Dropout(0.3),
        Bidirectional(LSTM(64, return_sequences=False)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='CNN_BiLSTM')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m




# ── Stronger variants ────────────────────────────────────────────────────
# Five additional architectures combining ideas that helped most in earlier
# LOUO sweeps: BatchNorm + GlobalAveragePooling, wider kernels, residual
# connections, and parallel multi-scale convolutions.

def build_bn_gap_wide(seq_len, n_chan, n_classes):
    """BN_GAP with wider Conv kernels — best out-of-the-box LOUO performer."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 8, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(64, 8, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(128, 5, padding='same'),
        BatchNormalization(), Activation('relu'),
        GlobalAveragePooling1D(),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='BN_GAP_Wide')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_widekernel_gap(seq_len, n_chan, n_classes):
    """WideKernel front-end with BN + GAP head (no Flatten) — ties BN_GAP_Wide."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(32, 8, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(64, 8, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(128, 5, padding='same'),
        BatchNormalization(), Activation('relu'),
        GlobalAveragePooling1D(),
        Dropout(0.3),
        Dense(n_classes, activation='softmax'),
    ], name='WideKernel_GAP')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_bn_gap_v2(seq_len, n_chan, n_classes):
    """Deeper/wider BN_GAP — Conv(64,5)+Conv(128,5)+Conv(128,3) + BN + GAP."""
    m = Sequential([
        Input(shape=(seq_len, n_chan)),
        Conv1D(64, 5, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(128, 5, padding='same'),
        BatchNormalization(), Activation('relu'),
        MaxPooling1D(2),
        Conv1D(128, 3, padding='same'),
        BatchNormalization(), Activation('relu'),
        GlobalAveragePooling1D(),
        Dropout(0.4),
        Dense(n_classes, activation='softmax'),
    ], name='BN_GAP_v2')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


from tensorflow.keras.layers import Add, Concatenate
from tensorflow.keras.models import Model

def build_resnet1d(seq_len, n_chan, n_classes):
    """Small 1D ResNet (3 residual blocks) with BN + GAP head."""
    inp = Input(shape=(seq_len, n_chan))
    x = inp
    for f in (64, 64, 128):
        shortcut = Conv1D(f, 1, padding='same')(x)
        h = Conv1D(f, 5, padding='same')(x)
        h = BatchNormalization()(h); h = Activation('relu')(h)
        h = Conv1D(f, 5, padding='same')(h)
        h = BatchNormalization()(h)
        x = Add()([shortcut, h])
        x = Activation('relu')(x)
        x = MaxPooling1D(2)(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.3)(x)
    out = Dense(n_classes, activation='softmax')(x)
    m = Model(inp, out, name='ResNet1D')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


def build_multiscale(seq_len, n_chan, n_classes):
    """Inception-like multi-scale CNN (kernels 3/5/8 in parallel per block)."""
    def block(x, f):
        b1 = Conv1D(f, 3, padding='same', activation='relu')(x)
        b2 = Conv1D(f, 5, padding='same', activation='relu')(x)
        b3 = Conv1D(f, 8, padding='same', activation='relu')(x)
        return Concatenate()([b1, b2, b3])
    inp = Input(shape=(seq_len, n_chan))
    x = block(inp, 16)
    x = BatchNormalization()(x); x = MaxPooling1D(2)(x)
    x = block(x, 32)
    x = BatchNormalization()(x); x = MaxPooling1D(2)(x)
    x = Conv1D(96, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.3)(x)
    out = Dense(n_classes, activation='softmax')(x)
    m = Model(inp, out, name='MultiScale')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m


VARIANTS = [
    ('Baseline',   'Conv1D(32,k=4)→Pool→Conv1D(64,k=4)→Pool→Flatten→Dense(64)→Dropout(0.3)→Softmax', build_baseline),
    ('Shallow',    'Single Conv block: Conv1D(32,k=5)→Pool→Flatten→Dense(32)→Dropout(0.3)→Softmax',           build_shallow),
    ('Deep',       'Three Conv blocks: Conv1D(32,4)→Conv1D(64,4)→Conv1D(128,3)→Flatten→Dense(128)→Dropout(0.4)→Softmax', build_deep),
    ('BN_GAP',     'Conv+BN+ReLU ×3 with GlobalAveragePooling1D head — fewer params, less overfit',                build_bn_gap),
    ('WideKernel', 'Conv1D(32,k=8)→Pool→Conv1D(64,k=8)→Pool→Flatten→Dense(64)→Dropout(0.3)→Softmax', build_wide_kernel),
    ('CNN_LSTM',   'Conv1D(32,4)→Pool→Conv1D(64,4)→Pool→Dropout→LSTM(64)→Dense(64)→Dropout→Softmax',           build_cnn_lstm),
    ('CNN_BiLSTM', 'Conv1D(32,4)→Pool→Conv1D(64,4)→Pool→Dropout→BiLSTM(64)→Dense(64)→Dropout→Softmax',         build_cnn_bilstm),
    # ── Tuned variants (LOUO leaders) ────────────────────────────────────────
    ('BN_GAP_Wide',    'Conv1D(32,k=8)+BN→Conv1D(64,k=8)+BN→Conv1D(128,k=5)+BN→GAP→Dropout(0.3)→Softmax  [LOUO leader]', build_bn_gap_wide),
    ('WideKernel_GAP', 'Same as BN_GAP_Wide (BN + wide kernels + GAP head)  [ties LOUO leader]',                       build_widekernel_gap),
    ('BN_GAP_v2',      'Conv1D(64,k=5)+BN→Conv1D(128,k=5)+BN→Conv1D(128,k=3)+BN→GAP→Dropout(0.4)→Softmax',              build_bn_gap_v2),
    ('ResNet1D',       '3 residual blocks (64,64,128 ch, k=5) + BN + GAP head',                                       build_resnet1d),
    ('MultiScale',     'Inception-like: parallel Conv k=3/5/8 in 2 blocks + Conv(96,3) + BN + GAP head',               build_multiscale),
]

print(f'Configured {len(VARIANTS)} architecture variant(s).')
for n, d, _ in VARIANTS:
    print(f'  - {n}')


Configured 12 architecture variant(s).
  - Baseline
  - Shallow
  - Deep
  - BN_GAP
  - WideKernel
  - CNN_LSTM
  - CNN_BiLSTM
  - BN_GAP_Wide
  - WideKernel_GAP
  - BN_GAP_v2
  - ResNet1D
  - MultiScale


## 3. Sensor column selection

Builds `SENSOR_COLS` — the ordered list of CSV column names that the model will read for every trial. The selection is driven by the boolean toggles in Section 2 sub-section 2: which hands to use (left / right), which body segments (wrist, palm, finger segments), and which signal types per segment (orientation, quaternion, accelerometer, flex).

The CSV schema is fixed, so column names are generated programmatically rather than parsed from each file header. If a configured column is missing from a particular CSV the loader simply skips that file without crashing.


In [17]:
SEGMENTS_WITH_FLEX = ['thumb', 'index', 'middle', 'ring', 'pinky']  # palm has no flex


def build_sensor_columns(hands, segments, use_ypr, use_quat, use_accel, use_flex):
    cols = []
    for hand in hands:
        for seg in segments:
            if seg == 'wrist':
                # wrist has a single IMU; YPR fields are heading/pitch/roll
                p = f'{hand}_wrist'
                if use_ypr:   cols += [f'{p}_heading', f'{p}_pitch', f'{p}_roll']
                if use_quat:  cols += [f'{p}_quat_w', f'{p}_quat_x', f'{p}_quat_y', f'{p}_quat_z']
                if use_accel: cols += [f'{p}_ax', f'{p}_ay', f'{p}_az']
                # no flex on wrist
            else:
                for loc in ['mid', 'prox']:
                    p = f'{hand}_{seg}_{loc}'
                    if use_ypr:   cols += [f'{p}_yaw', f'{p}_pitch', f'{p}_roll']
                    if use_quat:  cols += [f'{p}_quat_w', f'{p}_quat_x', f'{p}_quat_y', f'{p}_quat_z']
                    if use_accel: cols += [f'{p}_ax', f'{p}_ay', f'{p}_az']
                if use_flex and seg in SEGMENTS_WITH_FLEX:
                    cols += [f'{hand}_{seg}_mcp_flex', f'{hand}_{seg}_pip_flex']
    return cols


resolved_hands = [h for h, on in [('left', USE_LEFT_HAND), ('right', USE_RIGHT_HAND)] if on]
resolved_segs = [s for s, on in [
    ('wrist', USE_WRIST), ('palm', USE_PALM), ('thumb', USE_THUMB),
    ('index', USE_INDEX), ('middle', USE_MIDDLE), ('ring', USE_RING), ('pinky', USE_PINKY)
] if on]

SENSOR_COLS = build_sensor_columns(
    hands=resolved_hands, segments=resolved_segs,
    use_ypr=USE_YPR, use_quat=USE_QUAT, use_accel=USE_ACCEL, use_flex=USE_FLEX,
)

print(f'Selected {len(SENSOR_COLS)} sensor columns.')
print('First 6:', SENSOR_COLS[:6])
print('Last  6:', SENSOR_COLS[-6:])


Selected 176 sensor columns.
First 6: ['left_wrist_heading', 'left_wrist_pitch', 'left_wrist_roll', 'left_wrist_ax', 'left_wrist_ay', 'left_wrist_az']
Last  6: ['right_pinky_prox_roll', 'right_pinky_prox_ax', 'right_pinky_prox_ay', 'right_pinky_prox_az', 'right_pinky_mcp_flex', 'right_pinky_pip_flex']


## 3a. Preprocessing helpers

Two utilities used by the LOUO data loader (Section 6) to standardise raw trials before the network sees them:

- **`resample_trial`** — linearly resamples a `(T, C)` trial to `RESAMPLE_TO_N_STEPS` time steps. Gestures recorded at slightly different durations end up the same length, which is what the fixed-input CNN needs.
- **`apply_butterworth`** — zero-phase low-pass filter applied per channel. Strips high-frequency sensor noise while preserving the underlying motion. The cutoff frequency and filter order are set in Section 2.

Both functions are pure — they take trials in and return trials out, with no side effects.


In [18]:
def resample_trial(trial, n_steps):
    """Linear resample (T, C) → (n_steps, C)."""
    T, C = trial.shape
    if T == n_steps:
        return trial.astype(np.float32, copy=False)
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, n_steps)
    out = np.zeros((n_steps, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return out


def apply_butterworth(trials, cutoff, order, fs):
    """Zero-phase low-pass filter applied per channel."""
    nyq = fs / 2.0
    norm_cutoff = cutoff / nyq
    if norm_cutoff >= 1.0:
        print(f'  WARNING: cutoff {cutoff} Hz >= Nyquist {nyq} Hz — skipping filter.')
        return trials
    b, a = scipy_signal.butter(order, norm_cutoff, btype='low', analog=False)
    return [scipy_signal.filtfilt(b, a, t, axis=0).astype(np.float32) for t in trials]


# Resample


## 3b. Per-trial normalisation

The single biggest lever for cross-wearer accuracy. Each trial is rescaled along its time axis (one statistic per channel) using **only its own values**, so the absolute level and amplitude of every recording are removed while the *shape* of the motion is preserved.

### Why this matters

Two recordings of the same gesture from two different people typically differ in two ways that have nothing to do with the gesture itself:

- **DC offset** — their hands rest at slightly different angles, so gravity falls on each accelerometer channel at a different baseline.
- **Amplitude** — some people move more emphatically; larger hands move further; sensor sensitivities vary unit-to-unit.

If the model trains on raw values, it picks up those per-wearer traits as features. On a brand-new wearer the traits shift and the features stop working. Per-trial normalisation erases the absolute level and scale of every trial while preserving the temporal pattern.

### Modes

Selected by `PER_TRIAL_NORM` in Section 2 sub-section 6:

- **`'zscore'`** — each channel of each trial ends up with mean 0 and standard deviation 1.
- **`'minmax'`** — each channel of each trial is rescaled to `PER_TRIAL_MINMAX_RANGE` (default `(0, 1)`).
- **`'none'`** — skip this stage; the downstream Min-Max / Standard scaler is the only rescaling.

### Why it's leak-free

Each trial is normalised using only its own 90 time steps. No information crosses between trials, so nothing leaks from test back to train. When `PER_TRIAL_NORM != 'none'` the downstream scaler is automatically skipped to avoid double-normalising.


In [19]:
# Per-trial normalisation helpers. The dispatcher at the bottom
# (per_trial_normalise) is the only thing called by Section 6.
# The selector that controls which mode is used is PER_TRIAL_NORM
# in the Section 2 master config.

def per_trial_zscore(trials):
    """Z-score each trial along its time axis, per channel.

    Accepts a list of (T, C) arrays OR a stacked (N, T, C) array, and returns the
    same container type with each trial normalised to mean 0, std 1 along time.
    Constant channels are left at zero (std clamped by 1e-6).
    """
    def _zscore_one(t):
        mu = t.mean(axis=0, keepdims=True)                     # (1, C)
        sd = np.maximum(t.std(axis=0, keepdims=True), 1e-6)    # (1, C)
        return ((t - mu) / sd).astype(np.float32)
    if isinstance(trials, np.ndarray) and trials.ndim == 3:
        # Vectorised path for stacked arrays
        mu = trials.mean(axis=1, keepdims=True)
        sd = np.maximum(trials.std(axis=1, keepdims=True), 1e-6)
        return ((trials - mu) / sd).astype(np.float32)
    return [_zscore_one(t) for t in trials]

def per_trial_minmax(trials, out_range=(0.0, 1.0)):
    """MinMax-scale each trial along its time axis, per channel.

    For each (T, C) trial:
        x' = (x - x.min(time)) / max(x.max(time) - x.min(time), 1e-6)
        x' = x' * (hi - lo) + lo
    Same container in / out (list of (T,C) arrays or stacked (N,T,C) array).
    Constant channels (range == 0) collapse to the lower bound `lo`.
    """
    lo, hi = float(out_range[0]), float(out_range[1])
    span = hi - lo

    def _one(t):
        mn = t.min(axis=0, keepdims=True)                          # (1, C)
        rng = np.maximum(t.max(axis=0, keepdims=True) - mn, 1e-6)  # (1, C)
        return ((t - mn) / rng * span + lo).astype(np.float32)

    if isinstance(trials, np.ndarray) and trials.ndim == 3:
        mn = trials.min(axis=1, keepdims=True)
        rng = np.maximum(trials.max(axis=1, keepdims=True) - mn, 1e-6)
        return ((trials - mn) / rng * span + lo).astype(np.float32)
    return [_one(t) for t in trials]





def per_trial_normalise(trials, mode, minmax_range=(0.0, 1.0)):
    """Dispatcher used by the LOUO loader.

    mode: 'zscore' | 'minmax' | 'none'
    """
    if mode == 'zscore':
        return per_trial_zscore(trials)
    if mode == 'minmax':
        return per_trial_minmax(trials, out_range=minmax_range)
    if mode == 'none':
        return trials
    raise ValueError(f"PER_TRIAL_NORM must be 'zscore', 'minmax' or 'none', got {mode!r}")


## 4. Augmentation helpers

Defines a library of single-trial augmentations and the `augment_training_set(X, y, …)` driver that applies them. Each augmenter takes one `(T, C)` trial in and returns one trial out. The driver produces `AUGMENTATION_COPIES_PER_SAMPLE` augmented variants per original trial and concatenates them with the originals, so the model trains on (originals + their synthetic variants).

Each augmenter is gated by an entry in `AUGMENT_CONFIG` (Section 2 sub-section 4) with three common fields: `enabled` (on/off), `apply_prob` (chance of being applied to any given copy), and augmenter-specific parameters.

### Wearer-style augmenters

Three of the augmenters specifically simulate the differences between wearers — they are what makes augmentation useful for cross-wearer generalisation:

- **`rotate_accel`** — one random small 3-D rotation per trial applied to every accelerometer triple. Simulates the glove sitting at a slightly different angle on a new hand.
- **`channel_gain`** — independent random gain on every channel. Simulates per-sensor sensitivity differences and how forcefully a person performs the gestures.
- **`channel_dc`** — independent additive offset on every channel. Simulates the residual baseline difference between wearers.

Together these make every training epoch look like the model is seeing dozens of synthetic "wearers" per real one.

### Where augmentation runs

Augmentation is applied **only to the training pool inside each LOUO fold** — never to the held-out test user or to the inner-validation user. This keeps the evaluation honest while letting the model see a wider training distribution.


In [20]:
def _resize_linear(trial, target_steps):
    T, C = trial.shape
    if T == target_steps:
        return trial.astype(np.float32, copy=False)
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, target_steps)
    out = np.zeros((target_steps, C), dtype=np.float32)
    for c in range(C):
        out[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return out


def aug_time_shift(trial, max_shift_steps=3, fill_mode='edge', rng=None):
    rng = rng or np.random.default_rng()
    if max_shift_steps <= 0:
        return trial.copy()
    shift = int(rng.integers(-max_shift_steps, max_shift_steps + 1))
    if shift == 0:
        return trial.copy()
    out = np.empty_like(trial)
    out[:] = 0.0 if fill_mode == 'zero' else (trial[0] if shift > 0 else trial[-1])
    if shift > 0:
        out[shift:] = trial[:-shift]
    else:
        out[:shift] = trial[-shift:]
    return out


def aug_time_warp(trial, speed_range=(0.9, 1.1), rng=None):
    rng = rng or np.random.default_rng()
    factor = float(rng.uniform(*speed_range))
    T = trial.shape[0]
    warped_steps = max(4, int(round(T * factor)))
    return _resize_linear(_resize_linear(trial, warped_steps), T)


def aug_time_mask(trial, max_masks=2, max_mask_size=3, fill_mode='zero', rng=None):
    rng = rng or np.random.default_rng()
    out = trial.copy()
    T, C = out.shape
    n_masks = int(rng.integers(1, max_masks + 1)) if max_masks > 0 else 0
    for _ in range(n_masks):
        size = int(rng.integers(1, max_mask_size + 1))
        start = int(rng.integers(0, max(1, T - size + 1)))
        if fill_mode == 'mean':
            out[start:start+size] = out.mean(axis=0, keepdims=True)
        elif fill_mode == 'noise':
            ch_std = np.std(out, axis=0, keepdims=True)
            out[start:start+size] = rng.normal(0.0, np.maximum(ch_std, 1e-6), size=(size, C))
        else:
            out[start:start+size] = 0.0
    return out


def aug_gaussian_noise(trial, std_ratio=0.02, rng=None):
    rng = rng or np.random.default_rng()
    ch_std = np.std(trial, axis=0, keepdims=True)
    noise_std = np.maximum(ch_std * std_ratio, 1e-6)
    return (trial + rng.normal(0.0, noise_std, size=trial.shape)).astype(np.float32)


def aug_amplitude_scale(trial, scale_range=(0.95, 1.05), rng=None):
    rng = rng or np.random.default_rng()
    scale = rng.uniform(scale_range[0], scale_range[1], size=(1, trial.shape[1]))
    return (trial * scale).astype(np.float32)


def aug_baseline_offset(trial, offset_std_ratio=0.02, rng=None):
    rng = rng or np.random.default_rng()
    ch_std = np.std(trial, axis=0, keepdims=True)
    offset_std = np.maximum(ch_std * offset_std_ratio, 1e-6)
    offset = rng.normal(0.0, offset_std, size=(1, trial.shape[1]))
    return (trial + offset).astype(np.float32)


def aug_channel_dropout(trial, drop_fraction=0.03, fill_mode='zero', rng=None):
    rng = rng or np.random.default_rng()
    out = trial.copy()
    C = out.shape[1]
    n_drop = max(1, int(round(C * drop_fraction))) if drop_fraction > 0 else 0
    if n_drop == 0:
        return out
    idx = rng.choice(C, size=min(n_drop, C), replace=False)
    if fill_mode == 'mean':
        out[:, idx] = out.mean(axis=0, keepdims=True)[:, idx]
    else:
        out[:, idx] = 0.0
    return out


def aug_left_right_swap(trial, sensor_cols, rng=None):
    out = trial.copy()
    name_to_idx = {col: i for i, col in enumerate(sensor_cols)}
    for col in sensor_cols:
        if col.startswith('left'):
            partner = 'right' + col[4:]
            if partner in name_to_idx:
                i, j = name_to_idx[col], name_to_idx[partner]
                out[:, i], out[:, j] = trial[:, j], trial[:, i]
    return out



# ── Wearer-style augmenters — IMU rotation + per-channel gain/DC jitter ─
# These augmentations simulate the kind of inter-user variation we actually see
# in deployment: a glove sitting slightly off on a new wrist, sensor calibration
# drift, and per-channel DC bias. Applied to training trials ONLY (the augment
# loop never touches the hold-out or the LOUO test user).

def _find_accel_triples(sensor_cols):
    """Return list of (ax_idx, ay_idx, az_idx) column-index triples for every IMU.
    The dataset has 26 accel triples spread across both hands and all segments.
    """
    by_name = {c: i for i, c in enumerate(sensor_cols)}
    triples = []
    for prefix in sorted({c[:-3] for c in sensor_cols if c.endswith('_ax')}):
        ax, ay, az = f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az'
        if ax in by_name and ay in by_name and az in by_name:
            triples.append((by_name[ax], by_name[ay], by_name[az]))
    return triples

def _rotation_matrix(rx, ry, rz):
    """Build a 3x3 rotation matrix from Euler angles (radians, XYZ order)."""
    cx, sx = np.cos(rx), np.sin(rx)
    cy, sy = np.cos(ry), np.sin(ry)
    cz, sz = np.cos(rz), np.sin(rz)
    Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
    Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
    Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])
    return (Rz @ Ry @ Rx).astype(np.float32)

def aug_rotate_accel(trial, accel_triples, max_deg=10.0, rng=None):
    """Apply a SINGLE random small 3-D rotation to every accel triple in this trial.
    All triples share the rotation matrix — the physical analogue is the glove
    being seated a few degrees off, which rotates the whole hand frame at once.
    Drawing per-triple rotations would mean each fingertip's IMU is rotated
    independently, which is not a physical thing.
    """
    rng = rng or np.random.default_rng()
    if not accel_triples:
        return trial
    rad = np.deg2rad(max_deg)
    rx, ry, rz = rng.uniform(-rad, rad, size=3)
    R = _rotation_matrix(rx, ry, rz)
    out = trial.copy()
    for (ix, iy, iz) in accel_triples:
        v = out[:, [ix, iy, iz]]               # (T, 3)
        out[:, [ix, iy, iz]] = v @ R.T          # rotate every timestep by R
    return out

def aug_channel_gain(trial, scale_range=(0.85, 1.15), rng=None):
    """Multiply each channel by an independent random gain. Simulates per-user
    sensor calibration drift (different scale factors on different gloves).
    """
    rng = rng or np.random.default_rng()
    gains = rng.uniform(scale_range[0], scale_range[1],
                        size=(1, trial.shape[1])).astype(np.float32)
    return (trial * gains).astype(np.float32)

def aug_channel_dc(trial, offset_std_ratio=0.10, rng=None):
    """Add a per-channel DC offset (sigma = ratio × channel std). Simulates
    per-user sensor bias (different resting voltages).
    """
    rng = rng or np.random.default_rng()
    ch_std = np.std(trial, axis=0, keepdims=True)
    offsets = rng.normal(0.0, np.maximum(ch_std * offset_std_ratio, 1e-6),
                        size=(1, trial.shape[1])).astype(np.float32)
    return (trial + offsets).astype(np.float32)

def apply_augmentations(trial, sensor_cols, config, rng):
    """Apply each augmentation independently with its own probability."""
    out = trial.astype(np.float32, copy=True)

    def _drop_meta(cfg):
        return {k: v for k, v in cfg.items() if k not in ('enabled', 'apply_prob')}

    def _maybe(name, fn, **extra):
        cfg = config.get(name, {})
        if cfg.get('enabled', False) and rng.random() < cfg.get('apply_prob', 1.0):
            return fn(out, **_drop_meta(cfg), **extra, rng=rng)
        return out

    out = _maybe('time_shift',      aug_time_shift)
    out = _maybe('time_warp',       aug_time_warp)
    out = _maybe('time_mask',       aug_time_mask)
    out = _maybe('gaussian_noise',  aug_gaussian_noise)
    out = _maybe('amplitude_scale', aug_amplitude_scale)
    out = _maybe('baseline_offset', aug_baseline_offset)
    out = _maybe('channel_dropout', aug_channel_dropout)
    # Wearer-style augmenters (rotate / gain / DC)
    if config.get('rotate_accel', {}).get('enabled', False) \
            and rng.random() < config['rotate_accel'].get('apply_prob', 1.0):
        triples = config['rotate_accel'].get('_accel_triples') or _find_accel_triples(sensor_cols)
        out = aug_rotate_accel(out, triples,
                               max_deg=config['rotate_accel'].get('max_deg', 10.0), rng=rng)
    if config.get('channel_gain', {}).get('enabled', False) \
            and rng.random() < config['channel_gain'].get('apply_prob', 1.0):
        out = aug_channel_gain(out,
                               scale_range=config['channel_gain'].get('scale_range', (0.85, 1.15)),
                               rng=rng)
    if config.get('channel_dc', {}).get('enabled', False) \
            and rng.random() < config['channel_dc'].get('apply_prob', 1.0):
        out = aug_channel_dc(out,
                             offset_std_ratio=config['channel_dc'].get('offset_std_ratio', 0.10),
                             rng=rng)

    if config.get('left_right_swap', {}).get('enabled', False) \
            and rng.random() < config['left_right_swap'].get('apply_prob', 1.0):
        out = aug_left_right_swap(out, sensor_cols=sensor_cols, rng=rng)

    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def augment_training_set(X_tr, y_tr, sensor_cols, config, copies_per_sample=1, random_state=42):
    if copies_per_sample <= 0 or len(X_tr) == 0:
        return X_tr, y_tr
    rng = np.random.default_rng(random_state)
    X_aug, y_aug = [], []
    for trial, label in zip(X_tr, y_tr):
        for _ in range(copies_per_sample):
            X_aug.append(apply_augmentations(trial, sensor_cols, config, rng))
            y_aug.append(label)
    X_combined = np.concatenate([X_tr, np.stack(X_aug, axis=0)], axis=0)
    y_combined = np.concatenate([y_tr, np.array(y_aug, dtype=y_tr.dtype)], axis=0)
    return X_combined, y_combined


## 5. Downstream scaler helper

`scale_sets` fits a Min-Max or Standard scaler on the training pool and transforms the validation / test arrays with it. The scaler is fit on training data only, so no test-set statistics leak into preprocessing.

This stage runs **only when `PER_TRIAL_NORM = 'none'`** (Section 2 sub-section 6). When per-trial normalisation is on, the data is already on a common scale and a second rescaling would just shuffle the numbers around without adding information, so the LOUO loop and the final retrain skip this helper automatically.


In [21]:
# ── Downstream scaler ─────────────────────────────────────────────────────
# A single MinMax / Standard scaler fit on the training pool only and applied
# to the validation + test arrays. Used by the LOUO loop and the final retrain
# WHEN PER_TRIAL_NORM == 'none' (otherwise data is already on unit scale and
# this stage is skipped to avoid double-normalising).

def make_scaler(kind):
    """Return a fresh sklearn scaler matching the configured kind, or None."""
    if kind == 'standard':
        return StandardScaler()
    if kind == 'minmax':
        return MinMaxScaler()
    return None


def scale_sets(X_train, X_val, X_test, kind):
    """Fit a scaler on X_train only, then transform X_val and X_test.

    The scaler is fit on flattened (samples × time × channels) data so it
    behaves identically to the original notebook. Returns the transformed
    arrays plus the fitted scaler so it can be persisted alongside the model.
    """
    scaler = make_scaler(kind)
    if scaler is None:
        return X_train, X_val, X_test, None
    Nt, T, C = X_train.shape
    Xt = scaler.fit_transform(X_train.reshape(Nt, T * C)).reshape(Nt, T, C).astype(np.float32)
    Xv = (scaler.transform(X_val.reshape(X_val.shape[0], T * C))
              .reshape(X_val.shape[0], T, C).astype(np.float32)
          if len(X_val) else X_val)
    Xs = (scaler.transform(X_test.reshape(X_test.shape[0], T * C))
              .reshape(X_test.shape[0], T, C).astype(np.float32)
          if len(X_test) else X_test)
    return Xt, Xv, Xs, scaler


## 6. Leave-One-User-Out (LOUO) evaluation

The core measurement of the notebook. There is no other evaluation — the random train/test split and the k-fold cross-validation that earlier versions of this notebook had have been removed in favour of a single, honest cross-wearer test.

### What this cell does

1. **Load each user folder** listed in `LOUO_PATHS` (Section 2), applying the same preprocessing as the rest of the notebook — channel selection, resampling, Butterworth low-pass filter, and the configured per-trial normalisation.
2. **Run one fold per user.** In fold *k*, user *k*'s trials become the test set, and every other user forms the training pool.
3. **Select an inner-validation set** for EarlyStopping. With the default `LOUO_INNER_VAL = 'user'`, a deterministic rotation picks a *different* user from the training pool each fold to act as the validation user — never the held-out test user. This is what stops EarlyStopping from indirectly tuning on data from the wearer it will be scored on.
4. **Augment the remaining training users**, train the configured variant, and record the accuracy on the held-out user's trials.
5. **Report the mean ± std** across all folds for each variant in `LOUO_VARIANTS`. That number is what you should expect on a brand-new wearer.

Per-fold and per-variant results are collected into `louo_summary` (a list of dicts). Section 7 reads this to pick the variant to deploy; Section 8 serialises it into the experiment log; Section 9 renders it into the PDF report.


In [22]:
# ════════════════════════════════════════════════════════════════════════════
#  Section 4 — Leave-One-User-Out (LOUO) evaluation
#
#  All LOUO config lives in Section 2:
#      LOUO_PATHS              — list of <user>/Dynamic folders to include
#      LOUO_VARIANTS           — which architectures to evaluate
#      LOUO_INNER_VAL          — how to build the EarlyStopping val set per fold
#      AUGMENT_TRAINING_DATA   — apply AUGMENT_CONFIG to the training pool
#
#  This cell:
#      * loads each path into a per-user (X, y) dict
#      * runs one LOUO fold per user, per variant
#      * stores results in `louo_summary`
# ════════════════════════════════════════════════════════════════════════════
import glob as _glob, os as _os
from sklearn.metrics import accuracy_score as _accuracy_score
from sklearn.preprocessing import LabelEncoder as _LE
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping


# ──────────────────────────────────────────────────────────────────────────
#  Per-user loader. One call per LOUO_PATHS entry.
#  Re-uses the SAME preprocessing as Section 5 of the original pipeline
#  (resample → Butterworth → optional per-trial z-score).
# ──────────────────────────────────────────────────────────────────────────
def _louo_load_user(user_dir):
    """Read every CSV under <user_dir>/<label>/*.csv and preprocess it.

    Folders whose name is in EXCLUDE_CLASSES are skipped at this point —
    those trials never enter the pipeline.
    """
    trials, labels = [], []
    _excl = set(EXCLUDE_CLASSES)
    for label in sorted(_os.listdir(user_dir)):
        if label in _excl:
            continue   # blacklist: this class is dropped at load time
        ldir = _os.path.join(user_dir, label)
        if not _os.path.isdir(ldir):
            continue
        for fp in sorted(_glob.glob(_os.path.join(ldir, '*.csv'))):
            try:
                df = pd.read_csv(fp)
                avail = [c for c in SENSOR_COLS if c in df.columns]
                if not avail:
                    continue
                arr = df[avail].values.astype(np.float32)
                # Trim to the first USE_SECONDS of each gesture; recordings
                # shorter than the window are kept and stretched by resample.
                if _USE_N_SAMPLES < arr.shape[0]:
                    arr = arr[:_USE_N_SAMPLES]
                trials.append(arr)
                labels.append(label)
            except Exception as e:
                print(f'  ERR {fp}: {e}')
    if not trials:
        return None, None
    trials = [resample_trial(t, RESAMPLE_TO_N_STEPS) for t in trials]
    if APPLY_BUTTERWORTH:
        trials = apply_butterworth(trials, BUTTERWORTH_CUTOFF_HZ,
                                   BUTTERWORTH_ORDER, SAMPLING_RATE_HZ)
    # RAW = filtered + resampled, BEFORE per-trial normalisation — used by the
    # diagnostics page so cosine similarity / separation ratios stay in the
    # natural sensor scale (otherwise z-score makes every trial zero-mean and
    # cosine ≈ 0 for all classes).
    Xu_raw = np.stack(trials, axis=0).astype(np.float32)
    Xu_raw = np.nan_to_num(Xu_raw, nan=0.0, posinf=0.0, neginf=0.0)
    # Per-trial normalisation (z-score or min-max), Section 3b — used for model
    # training/evaluation.
    trials = per_trial_normalise(trials, PER_TRIAL_NORM, PER_TRIAL_MINMAX_RANGE)
    Xu = np.stack(trials, axis=0).astype(np.float32)
    Xu = np.nan_to_num(Xu, nan=0.0, posinf=0.0, neginf=0.0)
    return Xu, Xu_raw, np.array(labels)


# ──────────────────────────────────────────────────────────────────────────
#  Load every configured user. Key = user folder name (the parent dir of
#  the path you supplied), value = (X, y) for that user.
# ──────────────────────────────────────────────────────────────────────────
per_user_data = {}
per_user_data_raw = {}   # filtered + resampled, BEFORE per-trial norm (for diagnostics)
for p in LOUO_PATHS:
    p = str(p).rstrip('/')
    if not _os.path.isdir(p):
        print(f'  skip {p}  (folder not found)')
        continue
    user_name = _os.path.basename(_os.path.dirname(p))   # parent of <p>
    Xu, Xu_raw, yu = _louo_load_user(p)
    if Xu is None or len(Xu) == 0:
        print(f'  skip {user_name}  (no trials)')
        continue
    per_user_data[user_name] = (Xu, yu)
    per_user_data_raw[user_name] = (Xu_raw, yu)
    print(f'  loaded {user_name:<18s} {Xu.shape[0]} trials')
print(f'\nLoaded {len(per_user_data)} users for LOUO')
if EXCLUDE_CLASSES:
    print(f'Excluded classes (dropped at load time): {sorted(EXCLUDE_CLASSES)}')
if len(per_user_data) < 2:
    raise RuntimeError('LOUO needs at least 2 users with data — check LOUO_PATHS')

# Global label encoder across all users
all_labels = np.concatenate([per_user_data[u][1] for u in per_user_data])
le_louo = _LE().fit(all_labels)
n_classes_louo = len(le_louo.classes_)
print(f'Classes ({n_classes_louo}): {list(le_louo.classes_)}')

# Sequence + channel sizes inferred from the first loaded user
_first_user = next(iter(per_user_data))
sequence_length = per_user_data[_first_user][0].shape[1]
n_channels      = per_user_data[_first_user][0].shape[2]

# Builder lookup map: variant name -> build_<variant>(seq_len, n_chan, n_classes)
_builder_map = {n: b for n, _, b in VARIANTS}


# ──────────────────────────────────────────────────────────────────────────
#  LOUO loop
# ──────────────────────────────────────────────────────────────────────────
louo_summary = []
users_sorted = sorted(per_user_data)

for v_name in LOUO_VARIANTS:
    if v_name not in _builder_map:
        raise KeyError(f'Variant {v_name!r} not in VARIANTS '
                       f'(available: {list(_builder_map)})')
    builder = _builder_map[v_name]
    fold_accs = []
    fold_records = []
    print(f'\n── LOUO: {v_name}  (inner_val={LOUO_INNER_VAL}) ──')

    for fold_idx, held in enumerate(users_sorted):
        X_te_raw, y_te_str = per_user_data[held]

        # ── Choose the inner-validation source (per-user is recommended) ─
        other_users = [u for u in users_sorted if u != held]
        if LOUO_INNER_VAL == 'user':
            # Rotate the inner-val user per fold (deterministic; never == held)
            inner_val_user = other_users[fold_idx % len(other_users)]
            train_users = [u for u in other_users if u != inner_val_user]
        else:
            inner_val_user = None
            train_users = other_users

        # ── Build the training pool from the chosen train_users ──────────
        X_tr_raw = np.concatenate([per_user_data[u][0] for u in train_users], axis=0)
        y_tr_str = np.concatenate([per_user_data[u][1] for u in train_users], axis=0)
        y_tr     = le_louo.transform(y_tr_str)
        y_te     = le_louo.transform(y_te_str)

        # ── Augment training pool only ───────────────────────────────────
        if AUGMENT_TRAINING_DATA:
            X_tr_aug, y_tr_aug = augment_training_set(
                X_tr_raw, y_tr, sensor_cols=SENSOR_COLS, config=AUGMENT_CONFIG,
                copies_per_sample=AUGMENTATION_COPIES_PER_SAMPLE,
                random_state=AUGMENTATION_RANDOM_SEED,
            )
        else:
            X_tr_aug, y_tr_aug = X_tr_raw, y_tr

        # ── Build the validation set ─────────────────────────────────────
        if LOUO_INNER_VAL == 'user':
            X_val_raw, y_val_str = per_user_data[inner_val_user]
            y_val = le_louo.transform(y_val_str)
        elif LOUO_INNER_VAL == 'random':
            _rng = np.random.default_rng(RANDOM_STATE)
            _idx = _rng.permutation(len(X_tr_aug))
            _n_val = max(1, int(len(_idx) * LOUO_INNER_VAL_SPLIT))
            X_val_raw = X_tr_aug[_idx[:_n_val]]
            y_val     = y_tr_aug[_idx[:_n_val]]
            X_tr_aug  = X_tr_aug[_idx[_n_val:]]
            y_tr_aug  = y_tr_aug[_idx[_n_val:]]
        else:
            X_val_raw, y_val = None, None

        # ── Fit scaler on training pool; transform val + test ────────────
        # If per-trial z-score is on, data is already unit-scale → skip.
        if PER_TRIAL_NORM != 'none':
            X_tr_s, X_val_s, X_te_s, scaler = X_tr_aug, X_val_raw, X_te_raw, None
        else:
            X_tr_s, _, X_te_s, scaler = scale_sets(
                X_tr_aug, X_tr_aug[:0], X_te_raw, NORMALISATION)
            if X_val_raw is not None and scaler is not None:
                N, T, C = X_val_raw.shape
                X_val_s = scaler.transform(
                    X_val_raw.reshape(N, T*C)).reshape(N, T, C).astype(np.float32)
            else:
                X_val_s = X_val_raw

        # ── Build and train the model ────────────────────────────────────
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(RANDOM_STATE)
        model_louo = builder(sequence_length, n_channels, n_classes_louo)

        cbs = []
        if USE_EARLY_STOPPING:
            if X_val_s is not None:
                cbs.append(EarlyStopping(monitor='val_loss',
                                         patience=EARLY_STOPPING_PATIENCE,
                                         restore_best_weights=True, mode='min'))
            else:
                cbs.append(EarlyStopping(monitor='loss',
                                         patience=EARLY_STOPPING_PATIENCE,
                                         restore_best_weights=True, mode='min'))

        fit_kwargs = dict(epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH_SIZE,
                          verbose=0, callbacks=cbs)
        if X_val_s is not None:
            fit_kwargs['validation_data'] = (X_val_s, y_val)
        _history = model_louo.fit(X_tr_s, y_tr_aug, **fit_kwargs)

        # ── Score on the held-out user ───────────────────────────────────
        yp  = np.argmax(model_louo.predict(X_te_s, verbose=0), axis=1)
        acc = float(_accuracy_score(y_te, yp))
        fold_accs.append(acc)
        # Keep per-fold predictions and training history so the PDF report
        # (Section 9) can build aggregated per-class metrics + mean curves.
        fold_records.append({'held_out': held,
                             'inner_val_user': inner_val_user,
                             'n_test': int(len(y_te)),
                             'n_train': int(len(y_tr_aug)),
                             'acc': acc,
                             'y_true': y_te.tolist(),
                             'y_pred': yp.tolist(),
                             'history': {k: [float(v) for v in vs]
                                         for k, vs in _history.history.items()}})
        extra = f' (inner_val_user={inner_val_user})' if inner_val_user else ''
        print(f'  held-out={held:<18s} n_test={len(y_te):>3d} acc={acc:.4f}{extra}')

    mean_acc = float(np.mean(fold_accs))
    std_acc  = float(np.std(fold_accs))
    print(f'  LOUO mean acc = {mean_acc:.4f}  ± {std_acc:.4f}')
    louo_summary.append({'variant': v_name, 'mean': mean_acc, 'std': std_acc,
                         'inner_val': LOUO_INNER_VAL,
                         'per_trial_norm': PER_TRIAL_NORM,
                         'fold_records': fold_records})

print('\n=== LOUO summary ===')
for s in sorted(louo_summary, key=lambda r: -r['mean']):
    print(f'  {s["variant"]:<15s}  mean={s["mean"]:.4f}  ± {s["std"]:.4f}'
          f'  (inner_val={s["inner_val"]}, per_trial_norm={s["per_trial_norm"]})')


  loaded 1_Alan             30 trials
  loaded 2_Alex             30 trials
  loaded 3_Anghad           31 trials
  loaded 4_Daniel           30 trials
  loaded 6_Harry            30 trials
  loaded 8_Jestin           24 trials
  loaded 11_Mansh           30 trials
  loaded 12_Marcus          30 trials
  loaded 14_StephenV2       30 trials
  loaded 15_Tash            30 trials
  loaded 16_Bella           30 trials
  loaded 17_Raquel          30 trials
  loaded 18_Bridgette       30 trials

Loaded 13 users for LOUO
Excluded classes (dropped at load time): ['Double_Nothing']
Classes (6): [np.str_('Double_Lower'), np.str_('Double_Pistol_Recoil'), np.str_('Double_Raise'), np.str_('Double_Wiggle'), np.str_('Double_cmere'), np.str_('Drum_Roll')]

── LOUO: Baseline  (inner_val=user) ──


2026-05-14 13:42:10.840543: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:42:10.840599: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:42:10.840610: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:42:11.471917: I external/loca

  held-out=11_Mansh           n_test= 30 acc=0.9000 (inner_val_user=12_Marcus)
  held-out=12_Marcus          n_test= 30 acc=0.8667 (inner_val_user=14_StephenV2)
  held-out=14_StephenV2       n_test= 30 acc=0.8333 (inner_val_user=15_Tash)
  held-out=15_Tash            n_test= 30 acc=0.8333 (inner_val_user=16_Bella)
  held-out=16_Bella           n_test= 30 acc=0.8333 (inner_val_user=17_Raquel)
  held-out=17_Raquel          n_test= 30 acc=1.0000 (inner_val_user=18_Bridgette)
  held-out=18_Bridgette       n_test= 30 acc=0.6667 (inner_val_user=1_Alan)
  held-out=1_Alan             n_test= 30 acc=0.5000 (inner_val_user=2_Alex)


2026-05-14 13:44:37.832859: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:44:37.832883: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:44:37.832887: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:44:38.699127: I external/loca

  held-out=2_Alex             n_test= 30 acc=0.9667 (inner_val_user=3_Anghad)


...

  held-out=6_Harry            n_test= 30 acc=0.9667 (inner_val_user=8_Jestin)


2026-05-14 13:57:41.691008: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:57:42.423807: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_126', 8 bytes spill stores, 8 bytes spill loads

2026-05-14 13:57:42.506614: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_126', 240 bytes spill stores, 272 bytes spill loads



  held-out=8_Jestin           n_test= 24 acc=0.8750 (inner_val_user=11_Mansh)
  LOUO mean acc = 0.9033  ± 0.1309

── LOUO: WideKernel  (inner_val=user) ──


2026-05-14 13:57:49.880568: W external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2026-05-14 13:57:50.454935: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_362', 12 bytes spill stores, 12 bytes spill loads

2026-05-14 13:57:50.525640: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_362', 16 bytes spill stores, 16 bytes spill loads

2026-05-14 13:57:50.670285: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_362', 632 bytes

  held-out=11_Mansh           n_test= 30 acc=0.8333 (inner_val_user=12_Marcus)
  held-out=12_Marcus          n_test= 30 acc=0.8667 (inner_val_user=14_StephenV2)
  held-out=14_StephenV2       n_test= 30 acc=0.8333 (inner_val_user=15_Tash)
  held-out=15_Tash            n_test= 30 acc=0.7667 (inner_val_user=16_Bella)
  held-out=16_Bella           n_test= 30 acc=0.9667 (inner_val_user=17_Raquel)
  held-out=17_Raquel          n_test= 30 acc=0.8667 (inner_val_user=18_Bridgette)
  held-out=18_Bridgette       n_test= 30 acc=0.5000 (inner_val_user=1_Alan)
  held-out=1_Alan             n_test= 30 acc=0.2667 (inner_val_user=2_Alex)
  held-out=2_Alex             n_test= 30 acc=0.9667 (inner_val_user=3_Anghad)
  held-out=3_Anghad           n_test= 31 acc=1.0000 (inner_val_user=4_Daniel)
  held-out=4_Daniel           n_test= 30 acc=1.0000 (inner_val_user=6_Harry)
  held-out=6_Harry            n_test= 30 acc=0.9667 (inner_val_user=8_Jestin)
  held-out=8_Jestin           n_test= 24 acc=0.7083 (inner_v

## 7. Final retrain (deployment model)

LOUO measures how well the recipe generalises. This section trains **one** model using that recipe on the full data pool (optionally minus one held-out user) and exposes it as the deployment artefact.

Two settings drive this section, both in Section 2 sub-section 8:

- **`FINAL_RETRAIN_VARIANT`** — the architecture to save. Must be a name from `VARIANTS` (Section 2a).
- **`FINAL_RETRAIN_HELD_OUT_USER`** — name of one user to *exclude* from training, so the saved model still has a clean held-out accuracy reported alongside it. Set to `None` to train on every loaded user.

The same preprocessing, augmentation, and training settings used in LOUO are reused here. The resulting `model`, `history`, optional `scaler`, and `final_held_acc` are kept in memory for the next two sections.


In [23]:
# ═════════════════════════════════════════════════════════════════════════
#  Section 5 — Final retrain (deployment model)  [optional]
#
#  Controlled by RUN_FINAL_RETRAIN in Section 2. When False, this cell is a
#  no-op (no model is trained or saved), the LOUO results from Section 4
#  remain the deliverable, and downstream save / report cells short-circuit.
# ═════════════════════════════════════════════════════════════════════════
if not RUN_FINAL_RETRAIN:
    print('RUN_FINAL_RETRAIN=False — skipping final-retrain (Section 5).')
    # Expose the names that downstream cells (Section 6 save, PDF report) check
    # so they can short-circuit cleanly too.
    model = None
    scaler = None
    final_held_user = None
    final_held_acc  = None
    history = None
else:
    print(f'Final retrain variant : {FINAL_RETRAIN_VARIANT}')
    print(f'Held-out user         : {FINAL_RETRAIN_HELD_OUT_USER}')

    # ── Choose training users ─────────────────────────────────────────────────
    all_users = sorted(per_user_data)
    if FINAL_RETRAIN_HELD_OUT_USER is None:
        final_train_users = all_users
        final_held_user   = None
    else:
        if FINAL_RETRAIN_HELD_OUT_USER not in per_user_data:
            raise KeyError(f'FINAL_RETRAIN_HELD_OUT_USER={FINAL_RETRAIN_HELD_OUT_USER!r} '
                           f'not loaded. Available: {all_users}')
        final_train_users = [u for u in all_users if u != FINAL_RETRAIN_HELD_OUT_USER]
        final_held_user   = FINAL_RETRAIN_HELD_OUT_USER
    print(f'Training on           : {len(final_train_users)} users')

    # ── Build the training pool ───────────────────────────────────────────────
    X_train_pool = np.concatenate([per_user_data[u][0] for u in final_train_users], axis=0)
    y_train_pool_str = np.concatenate([per_user_data[u][1] for u in final_train_users], axis=0)
    y_train_pool = le_louo.transform(y_train_pool_str)
    print(f'Training pool         : {X_train_pool.shape}')

    # ── Augment ───────────────────────────────────────────────────────────────
    if AUGMENT_TRAINING_DATA:
        X_train_final, y_train_final = augment_training_set(
            X_train_pool, y_train_pool, sensor_cols=SENSOR_COLS, config=AUGMENT_CONFIG,
            copies_per_sample=AUGMENTATION_COPIES_PER_SAMPLE,
            random_state=AUGMENTATION_RANDOM_SEED,
        )
    else:
        X_train_final, y_train_final = X_train_pool.copy(), y_train_pool.copy()
    print(f'After augmentation    : {X_train_final.shape}')

    # ── Optional held-out user → evaluation set ───────────────────────────────
    if final_held_user is not None:
        X_held_raw, y_held_str = per_user_data[final_held_user]
        y_held = le_louo.transform(y_held_str)
    else:
        X_held_raw, y_held = None, None

    # ── Scale (skipped when per-trial z-score is on) ──────────────────────────
    if PER_TRIAL_NORM != 'none':
        X_train_s = X_train_final
        X_held_s  = X_held_raw
        scaler    = None
    else:
        if X_held_raw is not None:
            X_train_s, _, X_held_s, scaler = scale_sets(
                X_train_final, X_train_final[:0], X_held_raw, NORMALISATION)
        else:
            # No held-out user → fit scaler on training pool only; nothing to transform on the held side.
            X_train_s, _, _, scaler = scale_sets(
                X_train_final, X_train_final[:0], X_train_final[:0], NORMALISATION)
            X_held_s = None

    # ── Build + fit ───────────────────────────────────────────────────────────
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    builder = _builder_map[FINAL_RETRAIN_VARIANT]
    model = builder(sequence_length, n_channels, n_classes_louo)

    cbs = []
    if USE_EARLY_STOPPING:
        _has_val = (X_held_s is not None and len(X_held_s) > 0 and y_held is not None)
        cbs.append(EarlyStopping(monitor=('val_loss' if _has_val else 'loss'),
                                 patience=EARLY_STOPPING_PATIENCE,
                                 restore_best_weights=True, mode='min'))

    fit_kwargs = dict(epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH_SIZE,
                      verbose=TRAIN_VERBOSE, callbacks=cbs)
    if X_held_s is not None and len(X_held_s) > 0 and y_held is not None:
        fit_kwargs['validation_data'] = (X_held_s, y_held)
    history = model.fit(X_train_s, y_train_final, **fit_kwargs)
    print(f'Trainable params      : {model.count_params():,}')

    # ── Evaluate on the held-out user, if any ─────────────────────────────────
    if X_held_s is not None and len(X_held_s) > 0 and y_held is not None:
        yp = np.argmax(model.predict(X_held_s, verbose=0), axis=1)
        final_held_acc = float(_accuracy_score(y_held, yp))
        print(f'Held-out ({final_held_user}) accuracy: {final_held_acc:.4f}')
    else:
        final_held_acc = None

RUN_FINAL_RETRAIN=False — skipping final-retrain (Section 5).


## 8. Save model + experiment log

Persists the deployment model and appends a row to a long-running CSV log of every experiment you've run. Controlled by **`SAVE_DEPLOYMENT_MODEL`** in Section 2 sub-section 8 — set it to `False` to skip this section entirely while still running LOUO and the final retrain.

When enabled this cell writes:

- **`cnn_<variant>_<timestamp>.keras`** — the trained Keras model.
- **`labels_<variant>_<timestamp>.json`** — class names in the order used by the model's softmax output (so you can decode predictions later).
- **`scaler_<variant>_<timestamp>.joblib`** — the fitted scaler, but only when `PER_TRIAL_NORM = 'none'` (otherwise no scaler is fit).
- One row appended to **`experiment_results.csv`**, capturing every configuration value, the LOUO summary as JSON, and the paths to the saved artefacts. Each row is self-contained, so you can recover any past run by reading a single line.

Files are written under `MODEL_OUTPUT_DIR` (Section 2 sub-section 9).


In [24]:
if not RUN_FINAL_RETRAIN:
    print('RUN_FINAL_RETRAIN=False — skipping deployment-model save + CSV log (Section 6).')
else:
    # ════════════════════════════════════════════════════════════════════════════
    #  Section 6 — Save model + append experiment row to CSV log
    # ════════════════════════════════════════════════════════════════════════════
    if not SAVE_DEPLOYMENT_MODEL:
        print('SAVE_DEPLOYMENT_MODEL=False — skipping model save + CSV log.')
    else:
        import json
        from datetime import datetime
        import joblib

        def _to_jsonable(obj):
            if isinstance(obj, dict):  return {str(k): _to_jsonable(v) for k, v in obj.items()}
            if isinstance(obj, (list, tuple)): return [_to_jsonable(v) for v in obj]
            if isinstance(obj, np.ndarray):    return obj.tolist()
            if isinstance(obj, np.integer):    return int(obj)
            if isinstance(obj, np.floating):   return float(obj)
            if isinstance(obj, np.bool_):      return bool(obj)
            return obj

        def _stable_json(obj):
            return json.dumps(_to_jsonable(obj), sort_keys=True, separators=(',', ':'))

        # ── Save the deployment model + scaler ────────────────────────────────────
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        model_path  = MODEL_OUTPUT_DIR / f'cnn_{FINAL_RETRAIN_VARIANT}_{ts}.keras'
        scaler_path = MODEL_OUTPUT_DIR / f'scaler_{FINAL_RETRAIN_VARIANT}_{ts}.joblib'
        labels_path = MODEL_OUTPUT_DIR / f'labels_{FINAL_RETRAIN_VARIANT}_{ts}.json'
        model.save(model_path)
        if scaler is not None:
            joblib.dump(scaler, scaler_path)
        labels_path.write_text(json.dumps(list(le_louo.classes_)))
        print(f'Saved model   → {model_path}')
        print(f'Saved labels  → {labels_path}')
        if scaler is not None:
            print(f'Saved scaler  → {scaler_path}')

        # ── Build experiment row ──────────────────────────────────────────────────
        row = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'louo_paths_count': len(LOUO_PATHS),
            'louo_users':       _stable_json(sorted(per_user_data)),
            'sensor_cols_count': len(SENSOR_COLS),
            'sensor_cols':       _stable_json(SENSOR_COLS),
            'resample_to_n_steps':   RESAMPLE_TO_N_STEPS,
            'apply_butterworth':     bool(APPLY_BUTTERWORTH),
            'butterworth_cutoff_hz': BUTTERWORTH_CUTOFF_HZ,
            'butterworth_order':     BUTTERWORTH_ORDER,
            'sampling_rate_hz':      SAMPLING_RATE_HZ,
            'normalisation':         NORMALISATION,
            'excluded_classes': ';'.join(sorted(EXCLUDE_CLASSES)) if EXCLUDE_CLASSES else '',
            'per_trial_norm':              PER_TRIAL_NORM,
            'per_trial_minmax_range':      str(PER_TRIAL_MINMAX_RANGE),
            'random_state':          int(RANDOM_STATE),
            'augment_training_data': bool(AUGMENT_TRAINING_DATA),
            'augmentation_copies_per_sample': int(AUGMENTATION_COPIES_PER_SAMPLE),
            'augmentation_random_seed':       int(AUGMENTATION_RANDOM_SEED),
            'augmentation_config':  _stable_json(AUGMENT_CONFIG),
            'n_classes':            int(n_classes_louo),
            'class_names':          _stable_json(list(le_louo.classes_)),
            'sequence_length':      int(sequence_length),
            'n_channels':           int(n_channels),
            'epochs':               int(TRAIN_EPOCHS),
            'batch_size':           int(TRAIN_BATCH_SIZE),
            'early_stopping':       bool(USE_EARLY_STOPPING),
            'early_stopping_patience': int(EARLY_STOPPING_PATIENCE),
            'louo_inner_val':       LOUO_INNER_VAL,
            'louo_summary':         _stable_json(louo_summary),
            'final_retrain_variant':       FINAL_RETRAIN_VARIANT,
            'final_retrain_held_out_user': FINAL_RETRAIN_HELD_OUT_USER or '',
            'final_retrain_held_out_acc':  final_held_acc if final_held_acc is not None else '',
            'final_train_loss':            float(history.history['loss'][-1]),
            'final_train_accuracy':        float(history.history['accuracy'][-1]),
            'model_path':                  str(model_path),
            'scaler_path':                 str(scaler_path) if scaler is not None else '',
            'labels_path':                 str(labels_path),
        }
        # Per-variant summary columns (compact)
        for s in louo_summary:
            row[f'louo_{s["variant"]}_mean'] = s['mean']
            row[f'louo_{s["variant"]}_std']  = s['std']

        # ── Append to CSV log ─────────────────────────────────────────────────────
        log_df_row = pd.DataFrame([row])
        if EXPERIMENT_LOG_PATH.exists():
            log_df_row.to_csv(EXPERIMENT_LOG_PATH, mode='a', header=False, index=False)
        else:
            log_df_row.to_csv(EXPERIMENT_LOG_PATH, mode='w', header=True, index=False)
        print(f'Appended row to {EXPERIMENT_LOG_PATH}')

RUN_FINAL_RETRAIN=False — skipping deployment-model save + CSV log (Section 6).


## 9. PDF report

Renders a one-document summary of the run in the style of the earlier "variants_report" PDFs:

- **Title page** with every configuration value (paths, sensor channels, preprocessing, augmentation, training, evaluation strategy).
- **Variant comparison table** — params, LOUO mean ± std, fold count, and the optional held-out-user accuracy from the final retrain. The winning variant is tinted.
- **Bar chart** of LOUO accuracy per variant with error bars (std across folds).
- **Mean training curves** — accuracy and loss averaged across LOUO folds per variant, so you can spot under/overfitting trends without re-training anything.
- **One page per variant**: architecture description, params, LOUO summary, full per-fold table, aggregated per-class precision/recall/F1 across all folds, plus a second per-class table from the final retrain on the held-out user when that\'s configured.
- **Final page** with the deployment-model paths and final-retrain accuracy.

Controlled by **`GENERATE_PDF_REPORT`** in Section 2 sub-section 8 — independent of `SAVE_DEPLOYMENT_MODEL`. When the model isn\'t saved, the model-path fields show `— (not saved)` so the section still renders.

This report also includes a **Data diagnostics** page (within-class trial similarity, between-class separation ratio, t-SNE projection) and an **aggregated confusion matrix** per variant (raw counts and row-normalised side-by-side).

A **per-user diagnostics** page is also produced for every LOUO user (within-class similarity, between-class separation, and t-SNE) so you can spot users whose data is unusually noisy or whose classes overlap.


In [25]:
# ════════════════════════════════════════════════════════════════════════════
#  Section 9 — PDF report (variant comparison, charts, per-variant pages)
# ════════════════════════════════════════════════════════════════════════════
if not GENERATE_PDF_REPORT:
    print('GENERATE_PDF_REPORT=False — skipping PDF report.')
else:
    import io
    from datetime import datetime
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.lib.units import mm
    from reportlab.lib import colors
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak, Image,
    )

    # Fallbacks when SAVE_DEPLOYMENT_MODEL=False (those paths don't exist yet).
    model_path  = locals().get('model_path',  '— (not saved)')
    scaler_path = locals().get('scaler_path', '— (not saved)')
    labels_path = locals().get('labels_path', '— (not saved)')
    # Always stamp the PDF at the moment the report cell runs — so the filename
    # reflects when the report was generated, not when the model was saved.
    report_ts   = datetime.now()
    ts          = report_ts.strftime('%Y%m%d_%H%M%S')

    # ── Styles ────────────────────────────────────────────────────────────
    _TITLE = ParagraphStyle('TITLE', fontName='Helvetica-Bold', fontSize=18,
                            leading=22, spaceAfter=2,
                            textColor=colors.HexColor('#222222'))
    _META  = ParagraphStyle('META',  fontName='Helvetica',      fontSize=8,
                            leading=10, textColor=colors.HexColor('#666666'),
                            spaceAfter=8)
    _H1    = ParagraphStyle('H1',    fontName='Helvetica-Bold', fontSize=13,
                            leading=16, spaceBefore=10, spaceAfter=6,
                            textColor=colors.HexColor('#1f4e79'))
    _H2    = ParagraphStyle('H2',    fontName='Helvetica-BoldOblique', fontSize=10,
                            leading=12, spaceBefore=8, spaceAfter=3,
                            textColor=colors.HexColor('#1f4e79'))
    _BODY  = ParagraphStyle('BODY',  fontName='Helvetica',      fontSize=8,
                            leading=10)
    _KEY   = ParagraphStyle('KEY',   fontName='Helvetica-Bold', fontSize=8,
                            leading=10)

    def _wrap(text, style=_BODY):
        s = (str(text).replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;'))
        return Paragraph(s, style)

    def _kv_table(rows, widths=(55*mm, 130*mm)):
        body = [[_wrap(k, _KEY), _wrap(v)] for k, v in rows]
        t = Table(body, colWidths=widths, hAlign='LEFT')
        t.setStyle(TableStyle([
            ('VALIGN',         (0,0), (-1,-1), 'TOP'),
            ('LINEBELOW',      (0,0), (-1,-2), 0.25, colors.HexColor('#dddddd')),
            ('BOTTOMPADDING',  (0,0), (-1,-1), 2),
            ('TOPPADDING',     (0,0), (-1,-1), 2),
        ]))
        return t

    def _data_table(rows, col_widths, header_bg='#e8eef5', highlight_row=None,
                    highlight_bg='#fff3cd'):
        """Generic header + body table styled like the example report."""
        body = [[_wrap(c, _KEY if r == 0 else _BODY) for c in row]
                for r, row in enumerate(rows)]
        t = Table(body, colWidths=col_widths, hAlign='LEFT', repeatRows=1)
        style = [
            ('BACKGROUND',     (0,0), (-1,0),  colors.HexColor(header_bg)),
            ('GRID',           (0,0), (-1,-1), 0.25, colors.HexColor('#cccccc')),
            ('VALIGN',         (0,0), (-1,-1), 'MIDDLE'),
            ('BOTTOMPADDING',  (0,0), (-1,-1), 2),
            ('TOPPADDING',     (0,0), (-1,-1), 2),
        ]
        if highlight_row is not None and 0 < highlight_row < len(rows):
            style.append(('BACKGROUND', (0, highlight_row), (-1, highlight_row),
                          colors.HexColor(highlight_bg)))
        t.setStyle(TableStyle(style))
        return t

    # ── Helper — chart-to-Image flowable ─────────────────────────────────
    def _fig_to_image(fig, width_mm):
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image(buf, width=width_mm*mm,
                    height=width_mm*mm * fig.get_size_inches()[1] / fig.get_size_inches()[0])
        return img

    # ── Resolve which variant won + which to highlight ───────────────────
    _summary_sorted = sorted(louo_summary, key=lambda r: -r['mean'])
    _winner_name    = _summary_sorted[0]['variant'] if _summary_sorted else None

    # Optional held-out per-variant accuracy (only available for the variant
    # that actually got retrained — i.e. FINAL_RETRAIN_VARIANT).
    _held_out_acc_by_variant = {}
    if locals().get('final_held_acc') is not None and FINAL_RETRAIN_HELD_OUT_USER:
        _held_out_acc_by_variant[FINAL_RETRAIN_VARIANT] = final_held_acc

    # ── Build PDF ─────────────────────────────────────────────────────────
    pdf_path = MODEL_OUTPUT_DIR / f'report_{FINAL_RETRAIN_VARIANT}_{ts}.pdf'
    doc = SimpleDocTemplate(str(pdf_path), pagesize=A4,
                            leftMargin=15*mm, rightMargin=15*mm,
                            topMargin=15*mm, bottomMargin=15*mm)
    story = []

    # ── Title page ────────────────────────────────────────────────────────
    story.append(_wrap('Cross-Wearer Gesture Recognition — LOUO Variant Comparison', _TITLE))
    story.append(_wrap(f'Generated {report_ts.strftime("%Y-%m-%d %H:%M:%S")}', _META))

    # Data & configuration block
    story.append(_wrap('Data &amp; configuration', _H1))

    _config_rows = []
    for idx, p in enumerate(LOUO_PATHS, start=1):
        _config_rows.append((f'LOUO path {idx}', p))
    _config_rows += [
        ('LOUO users loaded', f'{len(per_user_data)} ({", ".join(sorted(per_user_data))})'),
        ('Excluded classes',
         ', '.join(sorted(EXCLUDE_CLASSES)) if EXCLUDE_CLASSES else '— (none)'),
        ('Classes', f'{n_classes_louo}: {", ".join(le_louo.classes_)}'),
        ('Sensor channels', f'{len(SENSOR_COLS)} (sequence length {sequence_length})'),
        ('Resample / filter',
         f'{RESAMPLE_TO_N_STEPS} steps · Butterworth cutoff '
         f'{BUTTERWORTH_CUTOFF_HZ} Hz, order {BUTTERWORTH_ORDER}'
         if APPLY_BUTTERWORTH else f'{RESAMPLE_TO_N_STEPS} steps · filter disabled'),
        ('Per-trial normalisation',
         f'{PER_TRIAL_NORM}'
         + (f' (range {PER_TRIAL_MINMAX_RANGE})' if PER_TRIAL_NORM == 'minmax' else '')),
        ('Downstream normalisation', f'{NORMALISATION}'),
        ('Sampling rate', f'{SAMPLING_RATE_HZ} Hz'),
        ('Random state', RANDOM_STATE),
        ('Augmentation',
         (f'on · copies={AUGMENTATION_COPIES_PER_SAMPLE} · '
          + ", ".join(k for k, v in AUGMENT_CONFIG.items() if v.get('enabled')))
         if AUGMENT_TRAINING_DATA else 'off'),
        ('Epochs · batch size', f'{TRAIN_EPOCHS} · {TRAIN_BATCH_SIZE}'),
        ('Early stopping',
         f'on · monitor val_loss · patience={EARLY_STOPPING_PATIENCE}'
         if USE_EARLY_STOPPING else 'off'),
        ('LOUO inner-validation', LOUO_INNER_VAL
         + (f' (split={LOUO_INNER_VAL_SPLIT})' if LOUO_INNER_VAL == 'random' else '')),
        ('LOUO variants', ', '.join(LOUO_VARIANTS)),
        ('Final-retrain variant', FINAL_RETRAIN_VARIANT),
        ('Final-retrain held-out user',
         FINAL_RETRAIN_HELD_OUT_USER if FINAL_RETRAIN_HELD_OUT_USER else '— (trained on all users)'),
    ]
    story.append(_kv_table(_config_rows))

    # ── Summary table ────────────────────────────────────────────────────
    story.append(_wrap('Variant comparison (LOUO)', _H1))

    summary_header = ['Variant', 'Params', 'LOUO mean', 'LOUO std',
                      'Folds', 'Held-out user acc']
    summary_rows = [summary_header]
    _builder_map_local = {n: b for n, _, b in VARIANTS}
    _winner_row_idx = None
    for r_idx, s in enumerate(_summary_sorted, start=1):
        # Build a dummy model only to count params (no fit, no eval).
        try:
            _m = _builder_map_local[s['variant']](sequence_length, n_channels, n_classes_louo)
            _params = f"{_m.count_params():,}"
            del _m
        except Exception:
            _params = '—'
        held_acc = _held_out_acc_by_variant.get(s['variant'])
        summary_rows.append([
            s['variant'],
            _params,
            f"{s['mean']:.4f}",
            f"{s['std']:.4f}",
            f"{len(s['fold_records'])}",
            f"{held_acc:.4f}" if held_acc is not None else '—',
        ])
        if s['variant'] == _winner_name:
            _winner_row_idx = r_idx

    story.append(_data_table(
        summary_rows,
        col_widths=[34*mm, 22*mm, 24*mm, 22*mm, 16*mm, 32*mm],
        highlight_row=_winner_row_idx,
    ))
    if _winner_name is not None:
        story.append(Spacer(1, 4))
        story.append(_wrap(f'Best LOUO mean: <b>{_winner_name}</b>', _META))

    # ── Bar chart: LOUO mean accuracy per variant with error bars ────────
    if _summary_sorted:
        story.append(_wrap('LOUO accuracy per variant', _H1))
        _names = [s['variant'] for s in _summary_sorted]
        _means = [s['mean']    for s in _summary_sorted]
        _stds  = [s['std']     for s in _summary_sorted]
        fig, ax = plt.subplots(figsize=(8, 4))
        bars = ax.bar(_names, _means, yerr=_stds, capsize=4,
                      color='#4f81bd', edgecolor='#1f4e79')
        # Mark the winner bar
        if _winner_name in _names:
            bars[_names.index(_winner_name)].set_color('#ed7d31')
        ax.set_ylim(0.0, 1.05)
        ax.set_ylabel('LOUO accuracy')
        ax.set_title('LOUO mean accuracy per architecture (error bars = std across folds)')
        for n, m in zip(_names, _means):
            ax.text(n, m + 0.02, f'{m:.2f}', ha='center', va='bottom', fontsize=8)
        ax.grid(axis='y', linestyle=':', alpha=0.5)
        plt.xticks(rotation=20, ha='right')
        story.append(_fig_to_image(fig, width_mm=180))

    # ── Mean training curves per variant (across LOUO folds) ─────────────
    story.append(_wrap('Mean training curves across LOUO folds', _H1))
    _has_history = any(
        any('history' in fr and fr['history'] for fr in s['fold_records'])
        for s in _summary_sorted
    )
    if not _has_history:
        story.append(_wrap('No per-fold history available (re-run LOUO to populate).'))
    else:
        # Aggregate per variant: pad each fold's history to the longest length,
        # then take the mean ignoring NaNs.
        def _mean_curve(records, key):
            seqs = [fr['history'].get(key, []) for fr in records if fr.get('history')]
            seqs = [s for s in seqs if s]
            if not seqs:
                return np.array([])
            L = max(len(s) for s in seqs)
            arr = np.full((len(seqs), L), np.nan)
            for i, s in enumerate(seqs):
                arr[i, :len(s)] = s
            return np.nanmean(arr, axis=0)

        fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(11, 4))
        for s in _summary_sorted:
            v_acc  = _mean_curve(s['fold_records'], 'val_accuracy')
            v_loss = _mean_curve(s['fold_records'], 'val_loss')
            t_acc  = _mean_curve(s['fold_records'], 'accuracy')
            t_loss = _mean_curve(s['fold_records'], 'loss')
            label = s['variant']
            if len(v_acc):
                ax_acc.plot(range(len(v_acc)), v_acc, label=f'{label}', linewidth=1.5)
            elif len(t_acc):
                ax_acc.plot(range(len(t_acc)), t_acc, label=f'{label} (train)',
                            linewidth=1.5, linestyle='--')
            if len(v_loss):
                ax_loss.plot(range(len(v_loss)), v_loss, label=f'{label}', linewidth=1.5)
            elif len(t_loss):
                ax_loss.plot(range(len(t_loss)), t_loss, label=f'{label} (train)',
                             linewidth=1.5, linestyle='--')
        for ax, ttl, ylab in [(ax_acc, 'Mean val accuracy (across LOUO folds)', 'Accuracy'),
                              (ax_loss, 'Mean val loss (across LOUO folds)', 'Loss')]:
            ax.set_title(ttl)
            ax.set_xlabel('Epoch')
            ax.set_ylabel(ylab)
            ax.grid(linestyle=':', alpha=0.5)
            ax.legend(fontsize=7, loc='best')
        story.append(_fig_to_image(fig, width_mm=180))


    # ── Diagnostics helper (used for pooled + per-user pages) ────────────
    # Renders three charts on the current `story`:
    #   1. within-class trial similarity (mean cosine, with min dots)
    #   2. between-class separation ratio heatmap
    #   3. t-SNE feature space projection
    # `X` is shape (N, T, C); `y` is shape (N,) of str labels.
    def _render_diag_block(story, X, y, subtitle):
        story.append(_wrap(subtitle, _META))
        if len(X) == 0:
            story.append(_wrap('(no trials)', _META))
            return
        classes = sorted(np.unique(y).tolist())
        feat = X.reshape(X.shape[0], -1)   # (N, T*C)

        # Within-class cosine similarity
        wc_mean, wc_min = [], []
        for c in classes:
            sub = feat[y == c]
            if len(sub) < 2:
                wc_mean.append(np.nan); wc_min.append(np.nan); continue
            sim = cosine_similarity(sub)
            iu = np.triu_indices_from(sim, k=1)
            vals = sim[iu]
            wc_mean.append(float(np.mean(vals)))
            wc_min.append(float(np.min(vals)))

        # Between-class separation: between-centroid distance / mean within-spread
        centroids = np.array([feat[y == c].mean(axis=0) for c in classes])
        within_spread = []
        for c in classes:
            sub = feat[y == c]
            within_spread.append(float(np.mean(pdist(sub))) if len(sub) >= 2 else np.nan)
        within_spread = np.array(within_spread)
        C = len(classes)
        ratio_mat = np.full((C, C), np.nan)
        for i in range(C):
            for j in range(C):
                if i == j:
                    continue
                bd = float(np.linalg.norm(centroids[i] - centroids[j]))
                ws = 0.5 * (within_spread[i] + within_spread[j])
                if np.isfinite(ws) and ws > 0:
                    ratio_mat[i, j] = bd / ws

        # ── Within-class similarity bar chart ────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 4))
        xp = np.arange(C)
        ax.bar(xp, wc_mean, color='#9ec3e6', edgecolor='#1f4e79',
               label='Mean similarity')
        ax.scatter(xp, wc_min, color='#c00000', zorder=5, label='Min similarity')
        ax.axhline(0.95, color='#daa520', ls='--', lw=1, label='0.95')
        ax.axhline(0.80, color='#c00000', ls='--', lw=1, label='0.80')
        for i, m in enumerate(wc_mean):
            if not np.isnan(m):
                ax.text(i, m + 0.01, f'{m:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_ylim(0.0, 1.05)
        ax.set_xticks(xp)
        ax.set_xticklabels(classes, rotation=20, ha='right')
        ax.set_ylabel('Cosine similarity')
        ax.set_title('Within-class trial similarity (1.0 = identical)')
        ax.legend(loc='lower right', fontsize=8)
        ax.grid(axis='y', linestyle=':', alpha=0.5)
        story.append(_fig_to_image(fig, width_mm=170))

        # ── Between-class separation ratio heatmap ───────────────────────
        fig, ax = plt.subplots(figsize=(7, 6))
        vmax = float(np.nanmax(ratio_mat)) if np.isfinite(np.nanmax(ratio_mat)) else 1.0
        im = ax.imshow(ratio_mat, cmap='RdYlGn', vmin=0.0, vmax=vmax)
        for i in range(C):
            for j in range(C):
                v = ratio_mat[i, j]
                if np.isnan(v):
                    continue
                # Adaptive formatting: 'Nx' for big ratios, decimals for small
                if v >= 10:    label = f'{v:.0f}x'
                elif v >= 1:   label = f'{v:.1f}x'
                else:          label = f'{v:.2f}'
                ax.text(j, i, label, ha='center', va='center',
                        fontsize=8, fontweight='bold', color='black')
        ax.set_xticks(range(C)); ax.set_yticks(range(C))
        ax.set_xticklabels(classes, rotation=35, ha='right')
        ax.set_yticklabels(classes)
        ax.set_title('Between-class separation ratio  (higher = easier to distinguish)')
        fig.colorbar(im, ax=ax, label='Separation ratio')
        story.append(_fig_to_image(fig, width_mm=170))

        # ── t-SNE feature space projection ───────────────────────────────
        if len(feat) >= 4:
            try:
                perp = max(2, min(30, max(2, len(feat) // 4)))
                tsne = TSNE(n_components=2, perplexity=perp,
                            random_state=RANDOM_STATE, init='pca',
                            learning_rate='auto')
                emb = tsne.fit_transform(feat)
                fig, ax = plt.subplots(figsize=(7, 6))
                palette = plt.cm.tab10(np.linspace(0, 1, max(C, 10)))
                for i, c in enumerate(classes):
                    msk = y == c
                    ax.scatter(emb[msk, 0], emb[msk, 1],
                               s=18, alpha=0.7, color=palette[i % len(palette)],
                               label=c, edgecolors='none')
                ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
                ax.set_title('t-SNE feature space projection')
                ax.legend(loc='best', fontsize=8, framealpha=0.85)
                ax.grid(linestyle=':', alpha=0.4)
                story.append(_fig_to_image(fig, width_mm=170))
            except Exception as e:
                story.append(_wrap(f'(t-SNE skipped: {e})', _META))
        else:
            story.append(_wrap('(t-SNE skipped — fewer than 4 trials)', _META))

    # ── Pooled diagnostics page (across all LOUO users) ──────────────────
    story.append(PageBreak())
    story.append(_wrap('Data diagnostics — pooled across users', _TITLE))
    # Use the RAW (un-normalised) trials so cosine/separation magnitudes
    # match what you'd see in a notebook print-out (~0.99 / 100x-500x).
    _pooled_X = np.concatenate([per_user_data_raw[u][0] for u in per_user_data_raw], axis=0)
    _pooled_y = np.concatenate([per_user_data_raw[u][1] for u in per_user_data_raw], axis=0)
    _render_diag_block(
        story, _pooled_X, _pooled_y,
        'Filtered + resampled trials from every LOUO user '
        '(before per-trial normalisation).',
    )
    story.append(_wrap(
        'Reading guide:  within-class similarity > 0.95 = very consistent; '
        '< 0.80 = high variance.  Separation ratio > 100x = trivially '
        'separable, 20-100x = easy, 5-20x = moderate, < 5x = hard '
        '(expect confusion).', _META))

    # ── Per-user diagnostics pages ───────────────────────────────────────
    for _u in sorted(per_user_data_raw):
        _Xu_raw, _yu = per_user_data_raw[_u]
        story.append(PageBreak())
        story.append(_wrap(f'Data diagnostics — user {_u}', _TITLE))
        _render_diag_block(
            story, _Xu_raw, _yu,
            f'{_Xu_raw.shape[0]} trials, {len(np.unique(_yu))} classes '
            f'(filtered + resampled, before per-trial normalisation).',
        )

    # ── Per-variant pages ────────────────────────────────────────────────
    _variant_desc = {n: d for n, d, _ in VARIANTS}

    for s in _summary_sorted:
        story.append(PageBreak())
        story.append(_wrap(f'Variant: {s["variant"]}', _TITLE))
        story.append(_wrap(_variant_desc.get(s['variant'], ''), _META))

        # Headline numbers
        try:
            _m = _builder_map_local[s['variant']](sequence_length, n_channels, n_classes_louo)
            _params = f"{_m.count_params():,}"
            del _m
        except Exception:
            _params = '—'
        story.append(_kv_table([
            ('Trainable params', _params),
            ('LOUO mean ± std', f"{s['mean']:.4f} ± {s['std']:.4f}"),
            ('Folds run',       len(s['fold_records'])),
            ('Inner-val mode',  s.get('inner_val', LOUO_INNER_VAL)),
            ('Per-trial norm',  s.get('per_trial_norm', PER_TRIAL_NORM)),
        ]))

        # Per-fold table (held-out user, n_train, n_test, accuracy)
        story.append(_wrap('LOUO per-fold results', _H2))
        fold_rows = [['Fold', 'Held-out user', 'Inner-val user', 'n_train', 'n_test', 'Accuracy']]
        for i, fr in enumerate(s['fold_records'], start=1):
            fold_rows.append([
                i,
                fr['held_out'],
                fr.get('inner_val_user') or '—',
                fr['n_train'],
                fr['n_test'],
                f"{fr['acc']:.4f}",
            ])
        story.append(_data_table(
            fold_rows,
            col_widths=[12*mm, 38*mm, 38*mm, 22*mm, 22*mm, 22*mm],
        ))

        # Aggregated per-class metrics across all folds
        story.append(_wrap('Per-class metrics (aggregated across folds)', _H2))
        _y_true_all = np.concatenate([np.array(fr['y_true']) for fr in s['fold_records']])                         if s['fold_records'] else np.array([])
        _y_pred_all = np.concatenate([np.array(fr['y_pred']) for fr in s['fold_records']])                         if s['fold_records'] else np.array([])
        if len(_y_true_all):
            _rep = classification_report(_y_true_all, _y_pred_all,
                                          target_names=list(le_louo.classes_),
                                          output_dict=True, zero_division=0)
            class_rows = [['Class', 'Precision', 'Recall', 'F1', 'Support']]
            for cls in list(le_louo.classes_) + ['macro avg', 'weighted avg']:
                r = _rep.get(cls)
                if r is None:
                    continue
                class_rows.append([
                    cls,
                    f"{r['precision']:.3f}",
                    f"{r['recall']:.3f}",
                    f"{r['f1-score']:.3f}",
                    int(r['support']),
                ])
            story.append(_data_table(
                class_rows,
                col_widths=[60*mm, 25*mm, 25*mm, 25*mm, 25*mm],
            ))

        # Aggregated confusion matrices (raw counts + row-normalised) ─────
        if len(_y_true_all):
            _cm = confusion_matrix(_y_true_all, _y_pred_all,
                                   labels=list(range(len(le_louo.classes_))))
            _cm_norm = _cm.astype(float)
            _row_sum = _cm_norm.sum(axis=1, keepdims=True)
            _cm_norm = np.divide(_cm_norm, _row_sum,
                                 out=np.zeros_like(_cm_norm),
                                 where=_row_sum > 0)
            _labels = list(le_louo.classes_)
            story.append(_wrap('Confusion matrix (aggregated across folds)', _H2))
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            for _ax, _M, _ttl, _cmap, _fmt in (
                (axes[0], _cm,      'Raw counts',           'Blues',  '{:d}'),
                (axes[1], _cm_norm, 'Row-normalised (recall)', 'Blues', '{:.2f}'),
            ):
                _im = _ax.imshow(_M, cmap=_cmap,
                                 vmin=0,
                                 vmax=_M.max() if _M.max() > 0 else 1)
                _ax.set_xticks(range(len(_labels)))
                _ax.set_yticks(range(len(_labels)))
                _ax.set_xticklabels(_labels, rotation=35, ha='right', fontsize=8)
                _ax.set_yticklabels(_labels, fontsize=8)
                _ax.set_xlabel('Predicted')
                _ax.set_ylabel('True')
                _ax.set_title(_ttl)
                _thr = _M.max() / 2 if _M.max() > 0 else 0
                for _i in range(_M.shape[0]):
                    for _j in range(_M.shape[1]):
                        _v = _M[_i, _j]
                        _color = 'white' if _v > _thr else 'black'
                        _ax.text(_j, _i, _fmt.format(_v),
                                 ha='center', va='center',
                                 fontsize=7, color=_color)
                fig.colorbar(_im, ax=_ax, fraction=0.046, pad=0.04)
            fig.tight_layout()
            story.append(_fig_to_image(fig, width_mm=180))


        else:
            story.append(_wrap('No predictions available.'))

        # Optional held-out (final retrain) per-class metrics
        if (s['variant'] == FINAL_RETRAIN_VARIANT
                and FINAL_RETRAIN_HELD_OUT_USER
                and locals().get('X_held_s') is not None
                and locals().get('y_held') is not None):
            try:
                _yp_held = np.argmax(model.predict(X_held_s, verbose=0), axis=1)
                story.append(_wrap(
                    f'Per-class metrics (final retrain, held-out user '
                    f'{FINAL_RETRAIN_HELD_OUT_USER})', _H2))
                _rep2 = classification_report(y_held, _yp_held,
                                               target_names=list(le_louo.classes_),
                                               output_dict=True, zero_division=0)
                rows2 = [['Class', 'Precision', 'Recall', 'F1', 'Support']]
                for cls in list(le_louo.classes_) + ['macro avg', 'weighted avg']:
                    r = _rep2.get(cls)
                    if r is None:
                        continue
                    rows2.append([
                        cls,
                        f"{r['precision']:.3f}",
                        f"{r['recall']:.3f}",
                        f"{r['f1-score']:.3f}",
                        int(r['support']),
                    ])
                story.append(_data_table(
                    rows2,
                    col_widths=[60*mm, 25*mm, 25*mm, 25*mm, 25*mm],
                ))
            except Exception as _e:
                story.append(_wrap(f'(held-out metrics unavailable: {_e})'))

    # ── Last page — final retrain artefacts ──────────────────────────────
    # story.append(PageBreak())
    # story.append(_wrap('Final retrain (deployment model)', _H1))
    # story.append(_kv_table([
    #     ('Variant',                FINAL_RETRAIN_VARIANT),
    #     ('Held-out user',          FINAL_RETRAIN_HELD_OUT_USER or '— (trained on all users)'),
    #     ('Held-out accuracy',
    #      f'{final_held_acc:.4f}' if locals().get('final_held_acc') is not None else '—'),
    #     ('Training-set size',
    #      f'{X_train_final.shape[0]} samples ({X_train_pool.shape[0]} before augmentation)'),
    #     ('Trainable params',       f'{model.count_params():,}'),
    #     ('Final train loss',       f'{history.history["loss"][-1]:.4f}'),
    #     ('Final train accuracy',   f'{history.history["accuracy"][-1]:.4f}'),
    #     ('Model path',             str(model_path)),
    #     ('Labels path',            str(labels_path)),
    #     ('Scaler path',
    #      str(scaler_path) if locals().get('scaler') is not None
    #      else '— (per-trial norm in use)'),
    # ]))

    doc.build(story)
    print(f'PDF report → {pdf_path}')


PDF report → /home/jestin/ThesisRepo/ML/NewReports/DynamicReports/report_BN_GAP_Wide_20260514_202319.pdf


## 10. Completion notification

Fires whichever of the channels in Section 2 sub-section 10 are enabled, so you can walk away while the run is in progress.

- **`NOTIFY_DESKTOP`** — Ubuntu toast (`notify-send`). Needs `libnotify-bin`, which is installed by default on most desktops.
- **`NOTIFY_BEEP`** — short system bell from inside the notebook (plays through whatever output device Jupyter has).
- **`NOTIFY_PUSH`** — off-device push via [ntfy.sh](https://ntfy.sh) (free, no signup). Install the ntfy app on your phone, subscribe to `NOTIFY_NTFY_TOPIC`, and you'll get a push the moment the run finishes. The topic acts like a password — pick something long and hard to guess.

The body includes the LOUO mean accuracy for each variant so you know the result before reopening the notebook. Each channel fails open: if it can't reach the desktop / network / speaker it prints a warning and the next channel still runs.


In [26]:
# ════════════════════════════════════════════════════════════════════════════
#  Section 10 — Completion notification
# ════════════════════════════════════════════════════════════════════════════
# Build the message body from the LOUO summary the eval cell just populated.
_summary_lines = ['LOUO complete.']
try:
    for _s in sorted(louo_summary, key=lambda r: -r['mean']):
        _summary_lines.append(
            f"  {_s['variant']}: {_s['mean']:.4f} ± {_s['std']:.4f}"
        )
except NameError:
    _summary_lines.append('  (no LOUO summary available)')
_notify_title = 'Gesture-recognition notebook'
_notify_body  = '\n'.join(_summary_lines)
print(_notify_body)


# ── Channel 1: Ubuntu desktop toast via notify-send ──────────────────────
if NOTIFY_DESKTOP:
    import shutil, subprocess
    if shutil.which('notify-send'):
        try:
            subprocess.run(
                ['notify-send', '--app-name=Jupyter',
                 '--icon=dialog-information', '--urgency=normal',
                 _notify_title, _notify_body],
                check=False,
            )
            print('Desktop notification sent.')
        except Exception as _e:
            print(f'Desktop notification failed: {_e}')
    else:
        print('Desktop notification skipped: notify-send not on PATH '
              '(install with `sudo apt install libnotify-bin`).')


# ── Channel 2: audible bell ───────────────────────────────────────────────
if NOTIFY_BEEP:
    try:
        # Three quick beeps separated by a short pause via the terminal BEL byte.
        import sys, time
        for _ in range(3):
            sys.stdout.write('\a')
            sys.stdout.flush()
            time.sleep(0.20)
        print('Beep sent.')
    except Exception as _e:
        print(f'Beep failed: {_e}')


# ── Channel 3: off-device push via ntfy.sh ────────────────────────────────
if NOTIFY_PUSH:
    try:
        import urllib.request
        if 'CHANGE-ME' in NOTIFY_NTFY_TOPIC:
            print('Push notification skipped: set NOTIFY_NTFY_TOPIC '
                  'in Section 2 sub-section (10) first.')
        else:
            req = urllib.request.Request(
                f'https://ntfy.sh/{NOTIFY_NTFY_TOPIC}',
                data=_notify_body.encode('utf-8'),
                headers={
                    'Title':    _notify_title,
                    'Priority': 'default',
                    'Tags':     'white_check_mark',
                },
                method='POST',
            )
            with urllib.request.urlopen(req, timeout=10) as _r:
                _ = _r.read()
            print(f'Push notification sent to ntfy topic {NOTIFY_NTFY_TOPIC!r}.')
    except Exception as _e:
        print(f'Push notification failed: {_e}')


LOUO complete.
  BN_GAP: 0.9033 ± 0.1309
  Deep: 0.8904 ± 0.1178
  Baseline: 0.8186 ± 0.1802
  WideKernel: 0.8109 ± 0.2070
  CNN_BiLSTM: 0.8083 ± 0.2229
  Shallow: 0.7692 ± 0.1928
  CNN_LSTM: 0.7628 ± 0.1861
Desktop notification sent.



(notify-send:1782506): libnotify-WARNING **: 20:25:24.343: Running in confined mode, using Portal notifications. Some features and hints won't be supported
libnotify-Message: 20:25:24.344: Category is not available when using Portal Notifications
libnotify-Message: 20:25:24.344: App Name is not available when using Portal Notifications


Beep sent.
